# Fisrt visit WGAN_GP . I explain that in detail. I skip some part here.

# Data extraction from .tar file

In [ ]:
# =======================
# 1. Extract BRATS Data
# =======================
import tarfile
import os

source_path = "/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar"
extract_path = "/kaggle/working/brats2021"

os.makedirs(extract_path, exist_ok=True)

with tarfile.open(source_path) as tar:
    tar.extractall(path=extract_path)

print("Extraction complete!")

In [ ]:
# ============================
# Display 20 Raw MRI Images
# Without Preprocessing
# ============================

import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt


extract_path = "/kaggle/working/brats2021"


# Get patient folders
patient_folders = sorted([
    f for f in os.listdir(extract_path)
    if os.path.isdir(os.path.join(extract_path, f))
    and "BraTS" in f
])


# Randomly select 20 patients
patients = np.random.choice(
    patient_folders,
    20,
    replace=False
)


plt.figure(figsize=(6,5))


for i, patient in enumerate(patients):

    patient_path = os.path.join(
        extract_path,
        patient
    )


    # Find FLAIR image
    flair_file = [
        f for f in os.listdir(patient_path)
        if "flair" in f.lower()
    ][0]


    flair_path = os.path.join(
        patient_path,
        flair_file
    )


    # Load original 3D MRI volume
    flair_volume = nib.load(flair_path).get_fdata()


    # Take middle axial slice
    slice_index = flair_volume.shape[2] // 2

    img = flair_volume[:, :, slice_index]


    plt.subplot(4,5,i+1)

    plt.imshow(
        img,
        cmap="gray"
    )

    plt.axis("off")


plt.suptitle(
    "Original BraTS 2021 FLAIR MRI Images Before Preprocessing",
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================
# Display and Save 20 Raw MRI Images
# Without Preprocessing
# 300 DPI for Research Paper
# ============================

import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

extract_path = "/kaggle/working/brats2021"

# Get patient folders
patient_folders = sorted([
    f for f in os.listdir(extract_path)
    if os.path.isdir(os.path.join(extract_path, f))
    and "BraTS" in f
])

# Randomly select 20 patients
patients = np.random.choice(
    patient_folders,
    12,
    replace=False
)

# Create figure
fig, axes = plt.subplots(
    3, 4,
    figsize=(10, 8)
)

for i, patient in enumerate(patients):

    patient_path = os.path.join(
        extract_path,
        patient
    )

    # Find FLAIR image
    flair_file = [
        f for f in os.listdir(patient_path)
        if "flair" in f.lower()
    ][0]

    flair_path = os.path.join(
        patient_path,
        flair_file
    )

    # Load original 3D MRI volume
    flair_volume = nib.load(flair_path).get_fdata()

    # Take middle axial slice
    slice_index = flair_volume.shape[2] // 2

    img = flair_volume[:, :, slice_index]

    # Display
    axes.flat[i].imshow(
        img,
        cmap="gray"
    )

    axes.flat[i].axis("off")


# Figure title
# fig.suptitle(
#     "Original BraTS 2021 FLAIR MRI Images Before Preprocessing",
#     fontsize=14
# )

# Adjust spacing
plt.tight_layout(rect=[0, 0, 1, 0.96])

# Save as 300 DPI PNG
save_path = "/kaggle/working/BraTS2021_raw_FLAIR_12_images_300dpi.png"

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    format="png"
)

plt.show()

print("Image saved successfully:")
print(save_path)

In [ ]:
# ============================
# Display and Save 12 Raw MRI Images
# 3 Rows × 4 Columns
# Compact Size for Research Paper
# 300 DPI
# ============================

import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

extract_path = "/kaggle/working/brats2021"

# Get patient folders
patient_folders = sorted([
    f for f in os.listdir(extract_path)
    if os.path.isdir(os.path.join(extract_path, f))
    and "BraTS" in f
])

# Randomly select 12 patients
patients = np.random.choice(
    patient_folders,
    12,
    replace=False
)

# Create compact figure
fig, axes = plt.subplots(
    3, 4,
    figsize=(6, 4.5)
)

for i, patient in enumerate(patients):

    patient_path = os.path.join(
        extract_path,
        patient
    )

    # Find FLAIR image
    flair_file = [
        f for f in os.listdir(patient_path)
        if "flair" in f.lower()
    ][0]

    flair_path = os.path.join(
        patient_path,
        flair_file
    )

    # Load original 3D MRI volume
    flair_volume = nib.load(flair_path).get_fdata()

    # Take middle axial slice
    slice_index = flair_volume.shape[2] // 2

    img = flair_volume[:, :, slice_index]

    # Display image
    axes.flat[i].imshow(
        img,
        cmap="gray"
    )

    axes.flat[i].axis("off")

# Remove unnecessary spacing
plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=0.02,
    hspace=0.02
)

# Save at 300 DPI
save_path = "/kaggle/working/BraTS2021_raw_FLAIR_12_images_300dpi2.png"

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    format="png"
)

plt.show()

print("Image saved successfully:")
print(save_path)

# Preprocess same as WGAN-GP. But filterning is slightly less than WGAN and DCGAN. We need more images for CGAN. since we are going to divided in two label. Small tumor and Large tumor images

In [ ]:
# ============================
# 1️⃣ Imports
# ============================
import os
import nibabel as nib
import numpy as np
import cv2
from tqdm import tqdm

IMG_SIZE = 128

extract_path = "/kaggle/working/brats2021"

input_images = []
tumor_sizes = []

# ============================
# 2️⃣ Normalize Function
# ============================
def normalize(img):
    img = img.astype(np.float32)
    img = (img - np.mean(img)) / (np.std(img)+1e-8)
    img = (img - img.min())/(img.max()-img.min()+1e-8)
    return img * 2 - 1

# ============================
# 3️⃣ Load ALL Patients
# ============================
patient_folders = sorted([
    f for f in os.listdir(extract_path)
    if os.path.isdir(os.path.join(extract_path, f)) and "BraTS" in f
])

print("Total patients:", len(patient_folders))

# ============================
# 4️⃣ Process Data
# ============================
for patient in tqdm(patient_folders):
    ppath = os.path.join(extract_path, patient)
    files = os.listdir(ppath)

    flair_path = [f for f in files if "flair" in f][0]
    seg_path   = [f for f in files if "seg" in f][0]

    flair_img = nib.load(os.path.join(ppath, flair_path)).get_fdata()
    seg_img   = nib.load(os.path.join(ppath, seg_path)).get_fdata()

    # Use middle slices only (avoid empty top/bottom)
    start = int(flair_img.shape[2] * 0.2)
    end   = int(flair_img.shape[2] * 0.8)

    for i in range(start, end):
        slice_img = flair_img[:, :, i]
        slice_seg = seg_img[:, :, i]

        # ------------------------
        # ❌ Remove empty slices
        # ------------------------
        if np.sum(slice_img > 0) < 500:
            continue

        # ------------------------
        # ❌ Remove very small tumor
        # ------------------------
        tumor_area = np.sum(slice_seg > 0)
        if tumor_area < 300:   # threshold (tune if needed)
            continue

        # ------------------------
        # Crop brain region
        # ------------------------
        coords = np.argwhere(slice_img > np.percentile(slice_img, 5))
        if coords.shape[0] == 0:
            continue

        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)

        pad = 10
        y_min = max(0, y_min - pad)
        x_min = max(0, x_min - pad)
        y_max = min(slice_img.shape[0], y_max + pad)
        x_max = min(slice_img.shape[1], x_max + pad)

        crop_img = slice_img[y_min:y_max, x_min:x_max]
        crop_seg = slice_seg[y_min:y_max, x_min:x_max]

        # ------------------------
        # Resize
        # ------------------------
        img = cv2.resize(crop_img, (IMG_SIZE, IMG_SIZE))
        segm = cv2.resize(crop_seg, (IMG_SIZE, IMG_SIZE))

        # ------------------------
        # Normalize
        # ------------------------
        img = normalize(img)

        input_images.append(img[..., None])
        tumor_sizes.append(np.sum(segm > 0))

# Convert to numpy
input_images = np.array(input_images, dtype="float32")
tumor_sizes = np.array(tumor_sizes)

print("Final dataset shape:", input_images.shape)

# Based on Median tumor value we split in two class. Each has 32k image. and DCGAN and WGAN has 41k. We try to balance as much as possible. But each GAN is different so necessary changes must take place.

In [ ]:
# ============================
# 1️⃣ Define Threshold
# ============================
# Use median tumor size as threshold
threshold = np.median(tumor_sizes)

# ============================
# 2️⃣ Split Dataset
# ============================
small_indices = np.where(tumor_sizes < threshold)[0]
large_indices = np.where(tumor_sizes >= threshold)[0]

small_images = input_images[small_indices]

large_images = input_images[large_indices]

# ============================
# 3️⃣ Print Counts
# ============================
print("Total Images:", len(input_images))
print("Small Tumor Images:", len(small_images))
print("Large Tumor Images:", len(large_images))

# Some images of both classes

In [ ]:
import matplotlib.pyplot as plt
import random

num_samples = 8

# Randomly pick samples
small_samples = random.sample(range(len(small_images)), num_samples)
large_samples = random.sample(range(len(large_images)), num_samples)

plt.figure(figsize=(16, 6))

# 🔹 Row 1: Small Tumors
for i, idx in enumerate(small_samples):
    img = (small_images[idx] + 1) / 2  # scale to [0,1]
    
    plt.subplot(2, num_samples, i+1)
    plt.imshow(img[:,:,0], cmap='gray')
    plt.title("Small")
    plt.axis('off')

# 🔹 Row 2: Large Tumors
for i, idx in enumerate(large_samples):
    img = (large_images[idx] + 1) / 2
    
    plt.subplot(2, num_samples, i+1+num_samples)
    plt.imshow(img[:,:,0], cmap='gray')
    plt.title("Large")
    plt.axis('off')

plt.suptitle("Top: Small Tumor | Bottom: Large Tumor (Big White Spot)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import random

num_samples = 8

# Randomly pick samples
small_samples = random.sample(range(len(small_images)), num_samples)
large_samples = random.sample(range(len(large_images)), num_samples)

plt.figure(figsize=(16, 6))

# 🔹 Row 1: Small Tumors
for i, idx in enumerate(small_samples):
    img = (small_images[idx] + 1) / 2  # scale to [0,1]
    
    plt.subplot(2, num_samples, i+1)
    plt.imshow(img[:,:,0], cmap='gray')
    plt.title("Small")
    plt.axis('off')

# 🔹 Row 2: Large Tumors
for i, idx in enumerate(large_samples):
    img = (large_images[idx] + 1) / 2
    
    plt.subplot(2, num_samples, i+1+num_samples)
    plt.imshow(img[:,:,0], cmap='gray')
    plt.title("Large")
    plt.axis('off')

plt.suptitle("Top: Small Tumor | Bottom: Large Tumor (Big White Spot)", fontsize=14)
plt.tight_layout()
plt.show()

# Use concatenate to merge both class and apply one-hot encoder

In [ ]:
import tensorflow as tf
import numpy as np
from IPython.display import clear_output


# Combine both datasets
input_images = np.concatenate([small_images, large_images], axis=0)

# Labels: 0 = small, 1 = large
labels_bin = np.array([0]*len(small_images) + [1]*len(large_images))

# Shuffle
idx = np.random.permutation(len(input_images))
input_images = input_images[idx]
labels_bin = labels_bin[idx]

# One-hot encode
labels_onehot = tf.one_hot(labels_bin, depth=2)

# Dataset
BATCH_SIZE = 32
dataset = tf.data.Dataset.from_tensor_slices((input_images, labels_onehot))
dataset = dataset.shuffle(5000).batch(BATCH_SIZE)

print("Final dataset:", input_images.shape)
print("Label count:", np.bincount(labels_bin))

# Generator same as WGAN-GP

In [ ]:
from tensorflow.keras import layers

z_dim = 100
num_classes = 2
IMG_SIZE = 128

def build_generator():
    noise = layers.Input(shape=(z_dim,))
    label = layers.Input(shape=(num_classes,))
    
    x = layers.Concatenate()([noise, label])

    x = layers.Dense(8*8*512, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Reshape((8,8,512))(x)

    x = layers.Conv2DTranspose(256,4,2,'same',use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(128,4,2,'same',use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(64,4,2,'same',use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    out = layers.Conv2DTranspose(1,4,2,'same',activation='tanh')(x)

    return tf.keras.Model([noise,label], out)

generator = build_generator()

# Discriminator Same


In [ ]:
def build_discriminator():
    img = layers.Input(shape=(IMG_SIZE,IMG_SIZE,1))
    label = layers.Input(shape=(IMG_SIZE,IMG_SIZE,2))

    x = layers.Concatenate()([img,label])

    x = layers.Conv2D(64,4,2,'same')(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(128,4,2,'same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(256,4,2,'same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(512,4,2,'same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    out = layers.Conv2D(1,4,1,'same')(x)

    return tf.keras.Model([img,label], out)

discriminator = build_discriminator()

# loss is BinaryCrossentropy
* learning rate G=1e-4,D=2e-4 and beta=0.5

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def gen_loss(fake):
    return bce(tf.ones_like(fake), fake)

def disc_loss(real, fake):
    return bce(tf.ones_like(real), real) + bce(tf.zeros_like(fake), fake)

g_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
d_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

# Training step

In [ ]:
@tf.function
def train_step(images, labels):
    batch = tf.shape(images)[0]
    noise = tf.random.normal([batch, z_dim])

    with tf.GradientTape() as gt, tf.GradientTape() as dt:
        fake = generator([noise, labels], training=True)

        label_map = tf.reshape(labels, [batch,1,1,2])
        label_map = tf.tile(label_map, [1,IMG_SIZE,IMG_SIZE,1])

        real_out = discriminator([images,label_map], training=True)
        fake_out = discriminator([fake,label_map], training=True)

        g_loss = gen_loss(fake_out)
        d_loss = disc_loss(real_out, fake_out)

    g_opt.apply_gradients(zip(gt.gradient(g_loss, generator.trainable_variables),
                              generator.trainable_variables))
    d_opt.apply_gradients(zip(dt.gradient(d_loss, discriminator.trainable_variables),
                              discriminator.trainable_variables))
    
    return g_loss, d_loss

In [ ]:
import matplotlib.pyplot as plt

def generate_images(epoch):
    noise = tf.random.normal([2, z_dim])
    labels = tf.one_hot([0,1], depth=2)  # small vs large

    imgs = generator([noise, labels], training=False)
    imgs = (imgs+1)/2

    plt.figure(figsize=(6,3))
    for i in range(2):
        plt.subplot(1,2,i+1)
        plt.imshow(imgs[i,:,:,0], cmap='gray')
        plt.title("Small Tumor" if i==0 else "Large Tumor")
        plt.axis('off')
    plt.suptitle(f"Epoch {epoch}")
    plt.show()

# Training loop
# each epoch time 4 min 24 sec

In [ ]:
from tqdm import tqdm

EPOCHS = 14
PATIENCE = 10

best_g_loss = np.inf
wait = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    g_losses = []
    d_losses = []

    for img_batch, lab_batch in tqdm(dataset):
        g_loss, d_loss = train_step(img_batch, lab_batch)

        g_losses.append(g_loss.numpy())
        d_losses.append(d_loss.numpy())

    avg_g = np.mean(g_losses)
    avg_d = np.mean(d_losses)
    
    clear_output(wait=True)

    print(f"G Loss: {avg_g:.4f} | D Loss: {avg_d:.4f}")

    # 🔥 Early stopping
    if avg_g < best_g_loss:
        best_g_loss = avg_g
        wait = 0
        generator.save("best_generator.h5")
        print("✅ Best model saved")
    else:
        wait += 1
        print(f"No improvement: {wait}/{PATIENCE}")

        if wait >= PATIENCE:
            print("🛑 Early stopping triggered")
            break

    generate_images(epoch+1)

# additional

In [ ]:
import tensorflow as tf
import numpy as np

generator = tf.keras.models.load_model(
    "/kaggle/working/best_generator.h5",
    compile=False
)

print("Generator loaded successfully!")

In [ ]:
NUM_IMAGES = 2000
HALF = NUM_IMAGES // 2

z_dim = 100

# Random noise
noise_small = tf.random.normal([HALF, z_dim])
noise_large = tf.random.normal([HALF, z_dim])

# Labels
label_small = tf.one_hot(np.zeros(HALF, dtype=np.int32), depth=2)
label_large = tf.one_hot(np.ones(HALF, dtype=np.int32), depth=2)

# Generate images
fake_small = generator([noise_small, label_small], training=False)
fake_large = generator([noise_large, label_large], training=False)

# Combine
generated_images = tf.concat([fake_small, fake_large], axis=0)

print("Generated shape:", generated_images.shape)

In [ ]:
generated_images = generated_images.numpy()

idx = np.random.permutation(len(generated_images))
generated_images = generated_images[idx]

In [ ]:
np.save("/kaggle/working/generated_images_2000.npy", generated_images)

print("Saved successfully!")

In [ ]:
imgs = np.load("/kaggle/working/generated_images_2000.npy")
print(imgs.shape)
print(imgs.min(), imgs.max())

In [ ]:
labels = np.concatenate([
    np.zeros(HALF, dtype=np.int32),   # small
    np.ones(HALF, dtype=np.int32)     # large
])

labels = labels[idx]

np.save("/kaggle/working/generated_labels_2000.npy", labels)

print("Images and labels saved.")

# Extra. Not Count

In [ ]:
TOTAL_IMAGES = 2000
BATCH_GEN = 32

all_images = []

steps = TOTAL_IMAGES // BATCH_GEN + int(TOTAL_IMAGES % BATCH_GEN != 0)

for step in range(steps):
    current_batch = min(BATCH_GEN, TOTAL_IMAGES - len(all_images)*BATCH_GEN)

    # -------------------------
    # Create balanced labels
    # -------------------------
    half = current_batch // 2
    labels = np.array([0]*half + [1]*(current_batch - half))
    np.random.shuffle(labels)

    labels_onehot = tf.one_hot(labels, depth=2)

    # -------------------------
    # Generate images
    # -------------------------
    noise = tf.random.normal([current_batch, z_dim])
    fake_images = generator([noise, labels_onehot], training=False)

    all_images.append(fake_images.numpy())

    print(f"Generated batch {step+1}/{steps}")

In [ ]:
generated_images = np.concatenate(all_images, axis=0)

print("Final shape:", generated_images.shape)

In [ ]:
np.save("generated_images_128.npy", generated_images)
print("✅ Saved successfully!")

In [ ]:
data = np.load("/kaggle/input/datasets/shantaakterid170/generated-cgan/generated_images_128 (8).npy")

print("Shape:", data.shape)
print("Min:", data.min(), "Max:", data.max())

# Extra. Not count. 

In [ ]:
indices = np.random.choice(len(data), 20, replace=False)

plt.figure(figsize=(10,8))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = data[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

# Generate 1000 small tumor and 1000 large tumor images

In [ ]:
TOTAL_IMAGES = 2000
BATCH_GEN = 32

all_images = []
all_labels = []   # ✅ ADD THIS

steps = TOTAL_IMAGES // BATCH_GEN + int(TOTAL_IMAGES % BATCH_GEN != 0)

generated_count = 0  # ✅ better tracking

for step in range(steps):
    current_batch = min(BATCH_GEN, TOTAL_IMAGES - generated_count)

    # -------------------------
    # Create balanced labels
    # -------------------------
    half = current_batch // 2
    labels = np.array([0]*half + [1]*(current_batch - half))
    np.random.shuffle(labels)

    labels_onehot = tf.one_hot(labels, depth=2)

    # -------------------------
    # Generate images
    # -------------------------
    noise = tf.random.normal([current_batch, z_dim])
    fake_images = generator([noise, labels_onehot], training=False)

    # ✅ STORE BOTH
    all_images.append(fake_images.numpy())
    all_labels.append(labels)

    generated_count += current_batch

    print(f"Generated batch {step+1}/{steps}")

# -------------------------
# Combine
# -------------------------
generated_images = np.concatenate(all_images, axis=0)
generated_labels = np.concatenate(all_labels, axis=0)

print("Images shape:", generated_images.shape)
print("Labels shape:", generated_labels.shape)

# -------------------------
# Save BOTH
# -------------------------
np.save("generated_images_128.npy", generated_images)
np.save("generated_labels.npy", generated_labels)

print("✅ Images + Labels saved successfully!")

# Generated images

In [ ]:
# new
import numpy as np
import matplotlib.pyplot as plt

# Load saved data
images = np.load("/kaggle/working/generated_images_2000.npy")
labels = np.load("/kaggle/working/generated_labels_2000.npy")

plt.figure(figsize=(10,6))

for i in range(20):
    plt.subplot(4,5,i+1)

    # Get image and normalize
    img = images[i].squeeze()
    img = (img + 1) / 2   # [-1,1] → [0,1]

    # Get label text
    label_text = "Small Tumor" if labels[i] == 0 else "Large Tumor"

    plt.imshow(img, cmap='gray')
    plt.title(label_text, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

# Extra. Not count. early stages images

In [ ]:
# new
generated_images = np.load("/kaggle/input/datasets/shantaakterid170/generated-cgan/generated_images_128 (8).npy")

indices = np.random.choice(len(generated_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = generated_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

In [ ]:

generated_images = np.load("/kaggle/working/brats2021")


indices = np.random.choice(len(generated_images), 20, replace=False)

plt.figure(figsize=(7,6))

for i, idx in enumerate(indices):
    plt.subplot(4,5,i+1)
    
    img = generated_images[idx].squeeze()
    img = (img + 1) / 2
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load saved data
images = np.load("generated_images_128.npy")
labels = np.load("generated_labels.npy")

plt.figure(figsize=(10,8))

for i in range(20):
    plt.subplot(4,5,i+1)

    # Get image and normalize
    img = images[i].squeeze()
    img = (img + 1) / 2   # [-1,1] → [0,1]

    # Get label text
    label_text = "Small Tumor" if labels[i] == 0 else "Large Tumor"

    plt.imshow(img, cmap='gray')
    plt.title(label_text, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Set seed for reproducibility (optional)
np.random.seed(42)

# Randomly select 2000 indices
idx = np.random.choice(len(input_images), 2000, replace=False)

real_subset = input_images[idx]
fake_images=images[:2000]

print("Subset shape:", real_subset.shape)

In [ ]:
real_images = real_subset   # from your preprocessing code

# fake_images = np.load("/kaggle/working/generated_images_128.npy")

print("Real:", real_images.shape)
print("Fake:", fake_images.shape)

In [ ]:
print("Shape:", fake_images.shape)
print("Min:", fake_images.min(), "Max:", fake_images.max())

In [ ]:
print("Shape:", real_images.shape)
print("Min:", real_images.min(), "Max:", real_images.max())

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input

def preprocess_images(images):
    # assume images are in [-1,1]
    images = (images + 1.0) * 127.5
    
    # grayscale → RGB
    images = np.repeat(images, 3, axis=-1)
    
    # resize to 299x299
    images = tf.image.resize(images, (299, 299)).numpy()
    
    return preprocess_input(images)

real_proc = preprocess_images(real_images)
fake_proc = preprocess_images(fake_images)

In [ ]:
from tensorflow.keras.applications import InceptionV3

model = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

In [ ]:
real_features = model.predict(real_proc, batch_size=32, verbose=1)
fake_features = model.predict(fake_proc, batch_size=32, verbose=1)

# FID score for best training.

In [ ]:
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# Inital FID score .not great.

In [ ]:
from scipy import linalg

def calculate_fid(f1, f2):
    mu1, sigma1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    mu2, sigma2 = f2.mean(axis=0), np.cov(f2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

fid_score = calculate_fid(real_features, fake_features)
print("🎯 FID Score:", fid_score)

# IS score

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input

def preprocess_images(images):
    # [-1,1] → [0,255]
    images = (images + 1.0) * 127.5
    
    # grayscale → RGB
    images = np.repeat(images, 3, axis=-1)
    
    # resize
    images = tf.image.resize(images, (299, 299)).numpy()
    
    return preprocess_input(images)

fake_proc = preprocess_images(fake_images)

In [ ]:
from tensorflow.keras.applications import InceptionV3

model = InceptionV3(include_top=True, weights='imagenet')

In [ ]:
preds = model.predict(fake_proc, batch_size=16, verbose=1)

In [ ]:
# new
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

In [ ]:
import numpy as np
from scipy.stats import entropy

def calculate_inception_score(preds, splits=10):
    scores = []
    N = preds.shape[0]
    
    for i in range(splits):
        part = preds[i * N // splits : (i + 1) * N // splits]
        
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        kl_sum = np.mean(np.sum(kl, axis=1))
        
        scores.append(np.exp(kl_sum))
    
    return np.mean(scores), np.std(scores)

is_mean, is_std = calculate_inception_score(preds)

print(f"🎯 Inception Score: {is_mean:.4f} ± {is_std:.4f}")

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 8 random images
idx_real = np.random.choice(len(real_images), 8, replace=False)
idx_fake = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[idx_real]
fake_sample = fake_images[idx_fake]

# Plot
plt.figure(figsize=(10, 8))

# -----------------------
# Real Images (Top Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, i + 1)
    plt.imshow(real_sample[i].squeeze(), cmap='gray')
    plt.title("Real")
    plt.axis('off')

# -----------------------
# Fake Images (Bottom Row)
# -----------------------
for i in range(8):
    plt.subplot(4, 4, 8 + i + 1)
    plt.imshow(fake_sample[i].squeeze(), cmap='gray')
    plt.title("Fake")
    plt.axis('off')

plt.tight_layout()
plt.show()

# Extra. Early stages

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 8 random images
idx_real = np.random.choice(len(real_images), 8, replace=False)
idx_fake = np.random.choice(len(fake_images), 8, replace=False)

real_sample = real_images[idx_real]
fake_sample = fake_images[idx_fake]

# Plot
plt.figure(figsize=(12, 6))

# -----------------------
# Real Images (Top Row)
# -----------------------
for i in range(8):
    plt.subplot(2, 8, i + 1)
    plt.imshow(real_sample[i].squeeze(), cmap='gray')
    plt.title("Real")
    plt.axis('off')

# -----------------------
# Fake Images (Bottom Row)
# -----------------------
for i in range(8):
    plt.subplot(2, 8, 8 + i + 1)
    plt.imshow(fake_sample[i].squeeze(), cmap='gray')
    plt.title("Fake")
    plt.axis('off')

plt.tight_layout()
plt.show()

# Some generated images

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(8,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(10,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(10,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Extra. Early stages images

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(fake_images), 20, replace=False)
samples = fake_images[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(10, 6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Data
epochs = [11, 12, 13, 14, 15, 16]
fid_scores = [91.49, 81.12, 84.87, 76.26, 83.10, 81.97]

# Plot
plt.figure()
plt.plot(epochs, fid_scores, marker='o')

plt.xlabel("Epoch")
plt.ylabel("FID Score")
plt.title("FID Score vs Epoch (WGAN)")

plt.grid()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Data
epochs = [11, 12, 13, 14, 15, 16]
is_scores = [1.9856, 2.0613, 2.1334, 2.1967, 2.2324, 2.0758]

# Plot
plt.figure()
plt.plot(epochs, is_scores, marker='o')

plt.xlabel("Epoch")
plt.ylabel("Inception Score")
plt.title("Inception Score vs Epoch (WGAN)")

plt.grid()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Data
epochs = [11, 12, 13, 14, 15, 16]
is_scores = [1.9856, 2.0613, 2.1334, 2.1967, 2.2324, 2.0758]

# Plot
plt.figure()
plt.plot(epochs, is_scores, marker='o')

plt.xlabel("Epoch")
plt.ylabel("Inception Score")
plt.title("Inception Score vs Epoch (WGAN)")

plt.grid()
plt.show()

In [ ]:
plt.savefig("fid_vs_epoch.png")
plt.savefig("is_vs_epoch.png")

In [ ]:
import matplotlib.pyplot as plt

models = ["DCGAN", "CGAN", "WGAN"]
fid_scores = [60.49, 34.06, 76.26]

plt.figure()
plt.bar(models, fid_scores)

plt.xlabel("Model")
plt.ylabel("FID Score")
plt.title("FID Comparison Across GAN Models")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

models = ["DCGAN", "CGAN", "WGAN"]
is_scores = [2.3527, 2.2573, 2.2324]

plt.figure()
plt.bar(models, is_scores)

plt.xlabel("Model")
plt.ylabel("Inception Score")
plt.title("Inception Score Comparison Across GAN Models")

plt.show()

# segmentation

In [ ]:
# ============================
# Imports
# ============================
import os
import nibabel as nib
import numpy as np
import cv2
from tqdm import tqdm

IMG_SIZE = 128
MAX_IMAGES = 4000

extract_path = "/kaggle/working/brats2021"

input_images = []
input_masks = []
tumor_sizes = []

# ============================
# Normalize Function
# ============================
def normalize(img):
    img = img.astype(np.float32)
    img = (img - np.mean(img)) / (np.std(img) + 1e-8)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img * 2 - 1

# ============================
# Patient folders
# ============================
patient_folders = sorted([
    f for f in os.listdir(extract_path)
    if os.path.isdir(os.path.join(extract_path, f)) and "BraTS" in f
])

print("Total patients:", len(patient_folders))

# ============================
# Process Images
# ============================
for patient in tqdm(patient_folders):

    if len(input_images) >= MAX_IMAGES:
        break

    ppath = os.path.join(extract_path, patient)
    files = os.listdir(ppath)

    flair_path = [f for f in files if "flair" in f][0]
    seg_path = [f for f in files if "seg" in f][0]

    flair_img = nib.load(os.path.join(ppath, flair_path)).get_fdata()
    seg_img = nib.load(os.path.join(ppath, seg_path)).get_fdata()

    start = int(flair_img.shape[2] * 0.2)
    end = int(flair_img.shape[2] * 0.8)

    for i in range(start, end):

        if len(input_images) >= MAX_IMAGES:
            break

        slice_img = flair_img[:, :, i]
        slice_seg = seg_img[:, :, i]

        # Remove empty slices
        if np.sum(slice_img > 0) < 500:
            continue

        # Remove very small tumors
        tumor_area = np.sum(slice_seg > 0)
        if tumor_area < 300:
            continue

        # Crop brain region
        coords = np.argwhere(slice_img > np.percentile(slice_img, 5))

        if len(coords) == 0:
            continue

        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)

        pad = 10

        y_min = max(0, y_min - pad)
        x_min = max(0, x_min - pad)
        y_max = min(slice_img.shape[0], y_max + pad)
        x_max = min(slice_img.shape[1], x_max + pad)

        crop_img = slice_img[y_min:y_max, x_min:x_max]
        crop_seg = slice_seg[y_min:y_max, x_min:x_max]

        # Resize image
        img = cv2.resize(
            crop_img,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_LINEAR
        )

        # Resize mask (VERY IMPORTANT)
        mask = cv2.resize(
            crop_seg,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_NEAREST
        )

        # Binary mask
        mask = (mask > 0).astype(np.float32)

        # Normalize MRI
        img = normalize(img)

        input_images.append(img[..., None])
        input_masks.append(mask[..., None])
        tumor_sizes.append(np.sum(mask))

# ============================
# Convert to numpy
# ============================
input_images = np.array(input_images, dtype=np.float32)
input_masks = np.array(input_masks, dtype=np.float32)
tumor_sizes = np.array(tumor_sizes)

print("Images :", input_images.shape)
print("Masks  :", input_masks.shape)
print("Tumors :", tumor_sizes.shape)

# Example:
# Images : (4000, 128, 128, 1)
# Masks  : (4000, 128, 128, 1)

In [ ]:
# ==========================================
# Display 12 Preprocessed Real MRI Images
# 3 Rows × 4 Columns
# 300 DPI for Research Paper
# ==========================================

import numpy as np
import matplotlib.pyplot as plt

# Randomly select 12 preprocessed MRI images
indices = np.random.choice(
    len(input_images),
    12,
    replace=False
)

# Create compact figure
fig, axes = plt.subplots(
    3, 4,
    figsize=(6, 4.5)
)

for ax, idx in zip(axes.flat, indices):

    # Remove channel dimension
    img = input_images[idx, :, :, 0]

    # Display preprocessed MRI
    ax.imshow(
        img,
        cmap="gray",
        vmin=-1,
        vmax=1
    )

    ax.axis("off")

# Tight spacing
plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=0.02,
    hspace=0.02
)

# Save 300 DPI image
save_path = "/kaggle/working/Real_MRI_Preprocessed_12_images_300dpi.png"

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    format="png"
)

plt.show()

print("Image saved successfully:")
print(save_path)

# Step 1: Split Real Data into Train / Validation / Test

In [ ]:
from sklearn.model_selection import train_test_split

# input_images = real MRI images (4000)
# input_masks  = real masks (4000)

# 80% Train+Val, 20% Test
X_trainval, X_test, Y_trainval, Y_test = train_test_split(
    input_images,
    input_masks,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# From Train+Val:
# 80% Train, 20% Validation

X_train, X_val, Y_train, Y_val = train_test_split(
    X_trainval,
    Y_trainval,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("Training   :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

# Step 2: Create Data Scarcity Dataset (10%)

In [ ]:
from sklearn.model_selection import train_test_split

X_real10, _, Y_real10, _ = train_test_split(
    X_train,
    Y_train,
    train_size=320,
    random_state=42,
    shuffle=True
)

print(X_real10.shape)
print(Y_real10.shape)

In [ ]:
import matplotlib.pyplot as plt

idx = np.random.randint(len(input_images))

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(input_images[idx].squeeze(), cmap='gray')
plt.title("MRI")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(input_masks[idx].squeeze(), cmap='gray')
plt.title("Mask")
plt.axis("off")

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Randomly select 16 images
indices = np.random.choice(len(input_images), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = input_images[idx].squeeze()
    mask = input_masks[idx].squeeze()

    # Display MRI
    ax.imshow(img, cmap='gray')

    # Overlay mask
    ax.imshow(mask, cmap='Reds', alpha=0.45)

    # Tumor size
    ax.set_title(f"Tumor Pixels: {int(mask.sum())}", fontsize=9)

    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Display 12 Real MRI Images with Tumor Mask
# 3 Rows × 4 Columns
# 300 DPI for Research Paper
# ==========================================

import matplotlib.pyplot as plt
import numpy as np

# Randomly select 12 images
indices = np.random.choice(
    len(input_images),
    12,
    replace=False
)

# Create compact figure
fig, axes = plt.subplots(
    3, 4,
    figsize=(6, 4.5)
)

for ax, idx in zip(axes.flat, indices):

    # Get MRI and corresponding mask
    img = input_images[idx].squeeze()
    mask = input_masks[idx].squeeze()

    # Display MRI
    ax.imshow(
        img,
        cmap="gray",
        vmin=-1,
        vmax=1
    )

    # Overlay tumor mask
    ax.imshow(
        mask,
        cmap="Reds",
        alpha=0.45
    )

    # Remove axes and labels
    ax.axis("off")

# Remove unnecessary spacing
plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=0.02,
    hspace=0.02
)

# Save at 300 DPI
save_path = "/kaggle/working/Real_MRI_with_Tumor_Masks_12_images_300dpi.png"

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    format="png"
)

plt.show()

print("Image saved successfully:")
print(save_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

indices = np.random.choice(len(input_images), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = input_images[idx].squeeze()
    mask = input_masks[idx].squeeze().astype(np.uint8)

    ax.imshow(img, cmap='gray')

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Draw contours in red
    for cnt in contours:
        cnt = cnt.squeeze()
        if len(cnt.shape) == 2:
            ax.plot(cnt[:, 0], cnt[:, 1], color='red', linewidth=2)

    ax.set_title(f"Image {idx}", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Display 12 Preprocessed Real MRI Images
# with Tumor Boundary Contours
# Same Size and Format as Previous Figures
# 3 Rows × 4 Columns
# 300 DPI
# ==========================================

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Randomly select 12 images
indices = np.random.choice(
    len(input_images),
    12,
    replace=False
)

# Same figure size as previous figures
fig, axes = plt.subplots(
    3, 4,
    figsize=(6, 4.5)
)

for ax, idx in zip(axes.flat, indices):

    # Get MRI and mask
    img = input_images[idx].squeeze()
    mask = input_masks[idx].squeeze().astype(np.uint8)

    # Display MRI
    ax.imshow(
        img,
        cmap="gray",
        vmin=-1,
        vmax=1
    )

    # Find tumor contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Draw tumor boundaries
    for cnt in contours:

        cnt = cnt.squeeze()

        if len(cnt.shape) == 2:
            ax.plot(
                cnt[:, 0],
                cnt[:, 1],
                color="red",
                linewidth=1.5
            )

    # Remove axes and labels
    ax.axis("off")

# Same spacing as previous figures
plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=0.02,
    hspace=0.02
)

# Save at 300 DPI
save_path = "/kaggle/working/Real_MRI_with_Tumor_Contours_12_images_300dpi.png"

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    format="png"
)

plt.show()

print("Image saved successfully:")
print(save_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Select 12 random images
indices = np.random.choice(len(input_images), 12, replace=False)

# Create 3 rows × 4 columns
fig, axes = plt.subplots(3, 4, figsize=(12, 9), dpi=300)

for ax, idx in zip(axes.ravel(), indices):

    img = input_images[idx].squeeze()
    mask = input_masks[idx].squeeze().astype(np.uint8)

    # Display MRI image
    ax.imshow(img, cmap='gray')

    # Find tumor contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Draw tumor contours
    for cnt in contours:
        cnt = cnt.squeeze()

        if len(cnt.shape) == 2:
            ax.plot(
                cnt[:, 0],
                cnt[:, 1],
                color='red',
                linewidth=2
            )

    # Remove axes, labels, and image numbers
    ax.axis('off')

plt.tight_layout(pad=0)

# Save at 300 DPI
save_path = "/kaggle/working/BraTS_MRI_Tumor_Examples_300dpi.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches='tight',
    pad_inches=0
)

plt.show()

print("Saved to:", save_path)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    input_images,
    input_masks,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

def conv_block(x, filters):

    x = Conv2D(filters,3,padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters,3,padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    return x

def encoder(x, filters):

    c = conv_block(x, filters)
    p = MaxPooling2D((2,2))(c)

    return c,p

def decoder(x, skip, filters):

    x = Conv2DTranspose(filters,2,strides=2,padding="same")(x)
    x = Concatenate()([x,skip])
    x = conv_block(x,filters)

    return x

def build_unet(input_shape=(128,128,1)):

    inputs = Input(input_shape)

    s1,p1 = encoder(inputs,64)
    s2,p2 = encoder(p1,128)
    s3,p3 = encoder(p2,256)
    s4,p4 = encoder(p3,512)

    b = conv_block(p4,1024)

    d1 = decoder(b,s4,512)
    d2 = decoder(d1,s3,256)
    d3 = decoder(d2,s2,128)
    d4 = decoder(d3,s1,64)

    outputs = Conv2D(1,1,activation="sigmoid")(d4)

    return Model(inputs,outputs)

model = build_unet()
model.summary()

In [ ]:
import tensorflow as tf

smooth = 1e-6

def dice_coef(y_true,y_pred):

    y_true=tf.reshape(y_true,[-1])
    y_pred=tf.reshape(y_pred,[-1])

    intersection=tf.reduce_sum(y_true*y_pred)

    return (2.*intersection+smooth)/(tf.reduce_sum(y_true)+tf.reduce_sum(y_pred)+smooth)

In [ ]:
def dice_loss(y_true,y_pred):

    return 1-dice_coef(y_true,y_pred)

In [ ]:
from tensorflow.keras.optimizers import Adam

def bce_dice_loss(y_true,y_pred):
    bce=tf.keras.losses.binary_crossentropy(y_true,y_pred)
    return bce+dice_loss(y_true,y_pred)

model.compile(
    optimizer=Adam(1e-4),
    loss=bce_dice_loss,
    metrics=[
        dice_coef,
        tf.keras.metrics.BinaryIoU(target_class_ids=[1])
    ]
)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

callbacks=[

    ModelCheckpoint(
        "best_unet.keras",
        monitor="val_dice_coef",
        mode="max",
        save_best_only=True
    ),

    EarlyStopping(
        monitor="val_dice_coef",
        patience=10,
        mode="max",
        restore_best_weights=True
    )
]

# history=model.fit(

#     X_train,
#     Y_train,

#     validation_data=(X_test,Y_test),

#     epochs=50,

#     batch_size=16,
 
#     callbacks=callbacks
# )

In [ ]:
model.evaluate(X_test,Y_test)

# Step 3: Train Baseline U-Net (320 Real Images)

In [ ]:
small_model = build_unet()

small_model.compile(
    optimizer='adam',
    loss=bce_dice_loss,
    metrics=[
        dice_coef,
        tf.keras.metrics.BinaryIoU(
            target_class_ids=[1]
        )
    ]
)

history_small = small_model.fit(
    X_real10,
    Y_real10,

    validation_data=(
        X_val,
        Y_val
    ),

    epochs=50,
    batch_size=16,

    callbacks=callbacks
)

# Step 4: Evaluate Baseline

In [ ]:
baseline_results = small_model.evaluate(
    X_test,
    Y_test,
    verbose=0
)

print("===== Baseline U-Net (320 Real Images) =====")
print(f"Loss : {baseline_results[0]:.4f}")
print(f"Dice : {baseline_results[1]:.4f}")
print(f"IoU  : {baseline_results[2]:.4f}")

# For 3200 images

In [ ]:
results = model.evaluate(X_test, Y_test, verbose=0)

loss = results[0]
dice = results[1]
iou = results[2]

print("="*40)
print("U-Net Evaluation on Test Set")
print("="*40)
print(f"Loss             : {loss:.4f}")
print(f"Dice Coefficient : {dice:.4f}")
print(f"Binary IoU       : {iou:.4f}")
print("="*40)

# Synthetic images of cGAN

In [ ]:
synthetic=np.load("/kaggle/input/datasets/shantaakterid170/generated-cgan/generated_images_128 (8).npy")

print(synthetic.shape)

# (2000,128,128,1)

In [ ]:
# ==========================================
# Display 12 cGAN Synthetic MRI Images
# 3 Rows × 4 Columns
# Same Size and Format as Previous Figures
# 300 DPI for Research Paper
# ==========================================

import numpy as np
import matplotlib.pyplot as plt

# Load cGAN synthetic images
synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/generated-cgan/generated_images_128 (8).npy"
)

print("Synthetic images shape:", synthetic.shape)

# Randomly select 12 images
indices = np.random.choice(
    len(synthetic),
    12,
    replace=False
)

# Same figure size as previous figures
fig, axes = plt.subplots(
    3, 4,
    figsize=(6, 4.5)
)

for ax, idx in zip(axes.flat, indices):

    # Remove channel dimension
    img = synthetic[idx].squeeze()

    # Display cGAN synthetic MRI
    ax.imshow(
        img,
        cmap="gray",
        vmin=-1,
        vmax=1
    )

    # Remove axes
    ax.axis("off")

# Same spacing as previous figures
plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=0.02,
    hspace=0.02
)

# Save at 300 DPI
save_path = "/kaggle/working/cGAN_Synthetic_MRI_12_images_300dpi.png"

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    format="png"
)

plt.show()

print("Image saved successfully:")
print(save_path)

In [ ]:
# ==========================================
# Display 12 cGAN Synthetic MRI Images
# with Red Tumor Contours
# 3 Rows × 4 Columns
# Same Size and Format as Previous Figures
# 300 DPI for Research Paper
# ==========================================

import cv2
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# Load cGAN Synthetic MRI Images
# ==========================================

synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/generated-cgan/generated_images_128 (8).npy"
)

# ==========================================
# Load Corresponding cGAN Synthetic Masks
# ==========================================

synthetic_masks = np.load(
    "/kaggle/input/datasets/shantaakterid170/synthetic-mask/synthetic_masks.npy"
)

print("Synthetic MRI shape :", synthetic.shape)
print("Synthetic mask shape:", synthetic_masks.shape)

# ==========================================
# Randomly Select 12 Images
# ==========================================

indices = np.random.choice(
    len(synthetic),
    12,
    replace=False
)

# ==========================================
# Create Figure
# ==========================================

fig, axes = plt.subplots(
    3, 4,
    figsize=(6, 4.5)
)

# ==========================================
# Display Images and Tumor Contours
# ==========================================

for ax, idx in zip(axes.flat, indices):

    # Get synthetic MRI
    img = synthetic[idx].squeeze()

    # Get corresponding synthetic mask
    mask = synthetic_masks[idx].squeeze()

    # Display synthetic MRI
    ax.imshow(
        img,
        cmap="gray",
        vmin=-1,
        vmax=1
    )

    # Convert mask to binary uint8
    mask = (mask > 0.5).astype(np.uint8)

    # Find tumor contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Draw tumor contours in red
    for cnt in contours:

        cnt = cnt.squeeze()

        if len(cnt.shape) == 2:

            ax.plot(
                cnt[:, 0],
                cnt[:, 1],
                color="red",
                linewidth=1.5
            )

    # Remove axes
    ax.axis("off")

# ==========================================
# Same Spacing as Previous Figures
# ==========================================

plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=0.02,
    hspace=0.02
)

# ==========================================
# Save at 300 DPI
# ==========================================

save_path = (
    "/kaggle/working/"
    "cGAN_Synthetic_MRI_with_Tumor_Contours_12_images_300dpi.png"
)

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    format="png"
)

plt.show()

print("Image saved successfully:")
print(save_path)

# Synthetic of DCGAN

In [ ]:
DCGAN_synthetic=np.load("/kaggle/input/datasets/shantaakterid170/dcgan-best-result/generated_images_DCGAN_BEST.npy")

print(DCGAN_synthetic.shape)

# (2000,128,128,1)

In [ ]:
# ==========================================
# Display 12 DCGAN Synthetic MRI Images
# Intensity Range: [0, 1]
# 3 Rows × 4 Columns
# Same Size and Format as Previous Figures
# 300 DPI for Research Paper
# ==========================================

import numpy as np
import matplotlib.pyplot as plt

# Load DCGAN synthetic images
DCGAN_synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/dcgan-best-result/generated_images_DCGAN_BEST.npy"
)

print("DCGAN synthetic images shape:", DCGAN_synthetic.shape)
print("Minimum intensity:", DCGAN_synthetic.min())
print("Maximum intensity:", DCGAN_synthetic.max())

# Randomly select 12 images
indices = np.random.choice(
    len(DCGAN_synthetic),
    12,
    replace=False
)

# Same figure size as previous figures
fig, axes = plt.subplots(
    3, 4,
    figsize=(6, 4.5)
)

for ax, idx in zip(axes.flat, indices):

    # Remove channel dimension
    img = DCGAN_synthetic[idx].squeeze()

    # Display DCGAN MRI
    ax.imshow(
        img,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    # Remove axes
    ax.axis("off")

# Same spacing as previous figures
plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=0.02,
    hspace=0.02
)

# Save at 300 DPI
save_path = (
    "/kaggle/working/"
    "DCGAN_Synthetic_MRI_12_images_300dpi.png"
)

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    format="png"
)

plt.show()

print("Image saved successfully:")
print(save_path)

In [ ]:
# ==========================================
# Display 12 DCGAN Synthetic MRI Images
# with Red Tumor Contours
# Using DCGAN_synthetic_for_mask.npy
# Intensity Range: [-1, 1]
# 3 Rows × 4 Columns
# Same Size and Format as Previous Figures
# 300 DPI for Research Paper
# ==========================================

import cv2
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# Load DCGAN Images Used for Mask Generation
# ==========================================

DCGAN_synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/pseudo-mask/DCGAN_synthetic_for_mask.npy"
)

# ==========================================
# Load Corresponding DCGAN Tumor Masks
# ==========================================

DCGAN_masks = np.load(
    "/kaggle/input/datasets/shantaakterid170/pseudo-mask/DCGAN_masks.npy"
)

print("DCGAN images shape:", DCGAN_synthetic.shape)
print("DCGAN masks shape :", DCGAN_masks.shape)

# ==========================================
# Randomly Select 12 Images
# ==========================================

indices = np.random.choice(
    len(DCGAN_synthetic),
    12,
    replace=False
)

# ==========================================
# Create Figure
# ==========================================

fig, axes = plt.subplots(
    3, 4,
    figsize=(6, 4.5)
)

# ==========================================
# Display MRI + Tumor Contours
# ==========================================

for ax, idx in zip(axes.flat, indices):

    # Get DCGAN synthetic MRI
    img = DCGAN_synthetic[idx].squeeze()

    # Get corresponding mask
    mask = DCGAN_masks[idx].squeeze()

    # Display DCGAN MRI
    ax.imshow(
        img,
        cmap="gray",
        vmin=-1,
        vmax=1
    )

    # Convert mask to binary
    mask = (mask > 0.5).astype(np.uint8)

    # Find tumor contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Draw tumor boundaries in red
    for cnt in contours:

        cnt = cnt.squeeze()

        if len(cnt.shape) == 2:
            ax.plot(
                cnt[:, 0],
                cnt[:, 1],
                color="red",
                linewidth=1.5
            )

    # Remove axes
    ax.axis("off")

# ==========================================
# Same Spacing as Previous Figures
# ==========================================

plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=0.02,
    hspace=0.02
)

# ==========================================
# Save at 300 DPI
# ==========================================

save_path = (
    "/kaggle/working/"
    "DCGAN_Synthetic_MRI_with_Tumor_Contours_12_images_300dpi.png"
)

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    format="png"
)

plt.show()

print("Image saved successfully:")
print(save_path)

# Synthetic WGAN

In [ ]:
WGAN_synthetic=np.load("/kaggle/input/datasets/shantaakterid170/wgan-best/WGANgenerated_images7.npy")

print(WGAN_synthetic.shape)

# (2000,128,128,1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load WGAN synthetic images
WGAN_synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/wgan-best/WGANgenerated_images7.npy"
)

print("WGAN images shape:", WGAN_synthetic.shape)

# Select 12 random images
indices = np.random.choice(len(WGAN_synthetic), 12, replace=False)

# Create 3 × 4 grid
fig, axes = plt.subplots(3, 4, figsize=(6, 4.5))

for ax, idx in zip(axes.flat, indices):
    img = WGAN_synthetic[idx].squeeze()

    ax.imshow(
        img,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    ax.axis("off")

# Minimal spacing
plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=0.02,
    hspace=0.02
)

# Save at 300 DPI
save_path = "/kaggle/working/WGAN_Synthetic_MRI_12_images_300dpi.png"

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    format="png"
)

plt.show()

print("Saved to:", save_path)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Load WGAN synthetic images
WGAN_synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/wgan-best/WGANgenerated_images7.npy"
)

# Load WGAN pseudo masks
WGAN_masks = np.load(
    "/kaggle/input/datasets/shantaakterid170/pseudo-mask/WGAN_masks.npy"
)

print("WGAN images shape:", WGAN_synthetic.shape)
print("WGAN masks shape :", WGAN_masks.shape)

# Select 12 random images
indices = np.random.choice(len(WGAN_synthetic), 12, replace=False)

# Create 3 × 4 grid
fig, axes = plt.subplots(3, 4, figsize=(6, 4.5))

for ax, idx in zip(axes.flat, indices):

    img = WGAN_synthetic[idx].squeeze()
    mask = WGAN_masks[idx].squeeze()

    # Display WGAN image [0, 1]
    ax.imshow(
        img,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    # Convert mask to binary
    mask = (mask > 0.5).astype(np.uint8)

    # Find tumor contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Draw tumor contours in red
    for cnt in contours:
        cnt = cnt.squeeze()

        if len(cnt.shape) == 2:
            ax.plot(
                cnt[:, 0],
                cnt[:, 1],
                color="red",
                linewidth=1.5
            )

    ax.axis("off")

# Minimal spacing
plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=0.02,
    hspace=0.02
)

# Save at 300 DPI
save_path = "/kaggle/working/WGAN_Synthetic_MRI_with_Tumor_Contours_12_images_300dpi.png"

fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
    format="png"
)

plt.show()

print("Saved to:", save_path)

# DCGAN Tumor mask distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. Real tumor masks
# ============================================================

input_masks = np.array(input_masks, dtype=np.float32)

# Select 2,000 real masks
np.random.seed(42)

real_indices = np.random.choice(
    len(input_masks),
    2000,
    replace=False
)

real_masks_2000 = input_masks[real_indices]

# Calculate tumor area
real_tumor_area = np.sum(
    real_masks_2000 > 0,
    axis=(1, 2, 3)
)


# ============================================================
# 2. DCGAN pseudo masks
# ============================================================

DCGAN_masks = np.load(
    "/kaggle/input/datasets/shantaakterid170/pseudo-mask/DCGAN_masks.npy"
)


# Use 2,000 DCGAN masks
DCGAN_masks = DCGAN_masks[:2000]
print("Real masks used :", len(real_tumor_area))

# Calculate tumor area
dcgan_tumor_area = np.sum(
    DCGAN_masks > 0.5,
    axis=(1, 2, 3)
)


# ============================================================
# 3. Statistics
# ============================================================
print("Real masks used :", len(real_tumor_area))
print("DCGAN masks used:", len(dcgan_tumor_area))

print("\n==============================")
print("Tumor Area Statistics")
print("==============================")

print("\nReal Masks:")
print("Number of masks :", len(real_tumor_area))
print("Minimum tumor pixels :", np.min(real_tumor_area))
print("Maximum tumor pixels :", np.max(real_tumor_area))
print("Mean tumor pixels    :", np.mean(real_tumor_area))
print("Median tumor pixels  :", np.median(real_tumor_area))
print("Std. deviation       :", np.std(real_tumor_area))

print("\nDCGAN Pseudo Masks:")
print("Number of masks :", len(dcgan_tumor_area))
print("Minimum tumor pixels :", np.min(dcgan_tumor_area))
print("Maximum tumor pixels :", np.max(dcgan_tumor_area))
print("Mean tumor pixels    :", np.mean(dcgan_tumor_area))
print("Median tumor pixels  :", np.median(dcgan_tumor_area))
print("Std. deviation       :", np.std(dcgan_tumor_area))


# ============================================================
# 4. Plot distribution
# ============================================================

plt.figure(figsize=(8, 5))

plt.hist(
    real_tumor_area,
    bins=50,
    alpha=0.5,
    label="Real Masks"
)

plt.hist(
    dcgan_tumor_area,
    bins=50,
    alpha=0.5,
    label="DCGAN Pseudo Masks"
)

plt.xlabel("Tumor pixels")
plt.ylabel("Number of images")
# plt.title("Tumor Area Distribution: Real vs. DCGAN")

plt.legend()

plt.tight_layout()


# ============================================================
# 5. Save at 300 DPI
# ============================================================

save_path = (
    "/kaggle/working/"
    "Tumor_Area_Distribution_Real_vs_DCGAN_300dpi.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05,
    format="png"
)

plt.show()

print("\nSaved to:")
print(save_path)

# cGAN

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. Real tumor masks
# ============================================================

input_masks = np.array(input_masks, dtype=np.float32)

# Select the same 2,000 real masks
np.random.seed(42)

real_indices = np.random.choice(
    len(input_masks),
    2000,
    replace=False
)

real_masks_2000 = input_masks[real_indices]

# Calculate tumor area
real_tumor_area = np.sum(
    real_masks_2000 > 0,
    axis=(1, 2, 3)
)


# ============================================================
# 2. cGAN pseudo masks
# ============================================================

cGAN_masks = np.load(
    "/kaggle/input/datasets/shantaakterid170/synthetic-mask/synthetic_masks.npy"
)

print("cGAN mask shape:", cGAN_masks.shape)

# Use 2,000 cGAN masks
cGAN_masks = cGAN_masks[:2000]

# Calculate tumor area
cgan_tumor_area = np.sum(
    cGAN_masks > 0.5,
    axis=(1, 2, 3)
)


# ============================================================
# 3. Statistics
# ============================================================

print("\n==============================")
print("Tumor Area Statistics")
print("==============================")

print("\nReal Masks:")
print("Number of masks       :", len(real_tumor_area))
print("Minimum tumor pixels  :", np.min(real_tumor_area))
print("Maximum tumor pixels  :", np.max(real_tumor_area))
print("Mean tumor pixels     :", np.mean(real_tumor_area))
print("Median tumor pixels   :", np.median(real_tumor_area))
print("Std. deviation        :", np.std(real_tumor_area))

print("\ncGAN Pseudo Masks:")
print("Number of masks       :", len(cgan_tumor_area))
print("Minimum tumor pixels  :", np.min(cgan_tumor_area))
print("Maximum tumor pixels  :", np.max(cgan_tumor_area))
print("Mean tumor pixels     :", np.mean(cgan_tumor_area))
print("Median tumor pixels   :", np.median(cgan_tumor_area))
print("Std. deviation        :", np.std(cgan_tumor_area))


# ============================================================
# 4. Plot distribution
# ============================================================

plt.figure(figsize=(8, 5))

plt.hist(
    real_tumor_area,
    bins=50,
    alpha=0.5,
    label="Real Masks"
)

plt.hist(
    cgan_tumor_area,
    bins=50,
    alpha=0.5,
    label="cGAN Pseudo Masks"
)

plt.xlabel("Tumor pixels")
plt.ylabel("Number of images")
# plt.title("Tumor Area Distribution: Real vs. cGAN")

plt.legend()

plt.tight_layout()


# ============================================================
# 5. Save at 300 DPI
# ============================================================

save_path = (
    "/kaggle/working/"
    "Tumor_Area_Distribution_Real_vs_cGAN_300dpi.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05,
    format="png"
)

plt.show()

print("\nSaved to:")
print(save_path)

# WGAN

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. Real tumor masks
# ============================================================

input_masks = np.array(input_masks, dtype=np.float32)

# Select the same 2,000 real masks
np.random.seed(42)

real_indices = np.random.choice(
    len(input_masks),
    2000,
    replace=False
)

real_masks_2000 = input_masks[real_indices]

# Calculate tumor area
real_tumor_area = np.sum(
    real_masks_2000 > 0,
    axis=(1, 2, 3)
)


# ============================================================
# 2. WGAN-GP pseudo masks
# ============================================================

WGAN_masks = np.load(
    "/kaggle/input/datasets/shantaakterid170/pseudo-mask/WGAN_masks.npy"
)

print("WGAN mask shape:", WGAN_masks.shape)

# Use 2,000 WGAN masks
WGAN_masks = WGAN_masks[:2000]

# Calculate tumor area
wgan_tumor_area = np.sum(
    WGAN_masks > 0.5,
    axis=(1, 2, 3)
)


# ============================================================
# 3. Statistics
# ============================================================

print("\n==============================")
print("Tumor Area Statistics")
print("==============================")

print("\nReal Masks:")
print("Number of masks       :", len(real_tumor_area))
print("Minimum tumor pixels  :", np.min(real_tumor_area))
print("Maximum tumor pixels  :", np.max(real_tumor_area))
print("Mean tumor pixels     :", np.mean(real_tumor_area))
print("Median tumor pixels   :", np.median(real_tumor_area))
print("Std. deviation        :", np.std(real_tumor_area))

print("\nWGAN-GP Pseudo Masks:")
print("Number of masks       :", len(wgan_tumor_area))
print("Minimum tumor pixels  :", np.min(wgan_tumor_area))
print("Maximum tumor pixels  :", np.max(wgan_tumor_area))
print("Mean tumor pixels     :", np.mean(wgan_tumor_area))
print("Median tumor pixels   :", np.median(wgan_tumor_area))
print("Std. deviation        :", np.std(wgan_tumor_area))


# ============================================================
# 4. Plot distribution
# ============================================================

plt.figure(figsize=(8, 5))

plt.hist(
    real_tumor_area,
    bins=50,
    alpha=0.5,
    label="Real Masks"
)

plt.hist(
    wgan_tumor_area,
    bins=50,
    alpha=0.5,
    label="WGAN-GP Pseudo Masks"
)

plt.xlabel("Tumor pixels")
plt.ylabel("Number of images")
# plt.title("Tumor Area Distribution: Real vs. WGAN-GP")

plt.legend()

plt.tight_layout()


# ============================================================
# 5. Save at 300 DPI
# ============================================================

save_path = (
    "/kaggle/working/"
    "Tumor_Area_Distribution_Real_vs_WGAN_300dpi.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05,
    format="png"
)

plt.show()

print("\nSaved to:")
print(save_path)

# PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# ============================================================
# 1. Select 2,000 real images
# ============================================================

input_images = np.array(input_images, dtype=np.float32)

np.random.seed(42)
real_indices = np.random.choice(
    len(input_images), 2000, replace=False
)

real_images = input_images[real_indices]


# ============================================================
# 2. Load 2,000 synthetic images
# ============================================================

DCGAN_synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/dcgan-best-result/generated_images_DCGAN_BEST.npy"
)[:2000]

cGAN_synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/generated-cgan/generated_images_128 (8).npy"
)[:2000]

WGAN_synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/wgan-best/WGANgenerated_images7.npy"
)[:2000]


# ============================================================
# 3. Convert all images to the SAME intensity range [-1, 1]
# ============================================================

# Real: already [-1, 1]
real_images = np.clip(real_images, -1, 1)

# cGAN: already [-1, 1]
cGAN_synthetic = np.clip(cGAN_synthetic, -1, 1)

# DCGAN: [0, 1] -> [-1, 1]
DCGAN_synthetic = DCGAN_synthetic * 2.0 - 1.0

# WGAN: [0, 1] -> [-1, 1]
WGAN_synthetic = WGAN_synthetic * 2.0 - 1.0


# ============================================================
# 4. Check intensity ranges
# ============================================================

print("Intensity ranges after normalization:")

print(
    "Real:   ",
    real_images.min(),
    "to",
    real_images.max()
)

print(
    "DCGAN:  ",
    DCGAN_synthetic.min(),
    "to",
    DCGAN_synthetic.max()
)

print(
    "cGAN:   ",
    cGAN_synthetic.min(),
    "to",
    cGAN_synthetic.max()
)

print(
    "WGAN:   ",
    WGAN_synthetic.min(),
    "to",
    WGAN_synthetic.max()
)


# ============================================================
# 5. Flatten images
# ============================================================

real_flat = real_images.reshape(2000, -1)

dcgan_flat = DCGAN_synthetic.reshape(2000, -1)

cgan_flat = cGAN_synthetic.reshape(2000, -1)

wgan_flat = WGAN_synthetic.reshape(2000, -1)


# ============================================================
# 6. Combine all datasets
# ============================================================

combined_data = np.concatenate(
    [
        real_flat,
        dcgan_flat,
        cgan_flat,
        wgan_flat
    ],
    axis=0
)

print("Combined data shape:", combined_data.shape)


# ============================================================
# 7. PCA
# ============================================================

pca = PCA(
    n_components=2,
    random_state=42
)

pca_result = pca.fit_transform(combined_data)


# ============================================================
# 8. Separate PCA results
# ============================================================

real_pca = pca_result[0:2000]

dcgan_pca = pca_result[2000:4000]

cgan_pca = pca_result[4000:6000]

wgan_pca = pca_result[6000:8000]


# ============================================================
# 9. Plot PCA
# ============================================================

plt.figure(figsize=(12, 10), dpi=300)

plt.scatter(
    real_pca[:, 0],
    real_pca[:, 1],
    c="#1f77b4",
    s=15,
    alpha=0.55,
    label="Real"
)

plt.scatter(
    dcgan_pca[:, 0],
    dcgan_pca[:, 1],
    c="#d62728",
    s=15,
    alpha=0.55,
    label="DCGAN"
)

plt.scatter(
    cgan_pca[:, 0],
    cgan_pca[:, 1],
    c="#2ca02c",
    s=15,
    alpha=0.55,
    label="cGAN"
)

plt.scatter(
    wgan_pca[:, 0],
    wgan_pca[:, 1],
    c="#ff7f0e",
    s=15,
    alpha=0.55,
    label="WGAN-GP"
)

plt.xlabel(
    f"PC1 ({100 * pca.explained_variance_ratio_[0]:.2f}%)",
    fontsize=13
)

plt.ylabel(
    f"PC2 ({100 * pca.explained_variance_ratio_[1]:.2f}%)",
    fontsize=13
)

plt.legend(
    frameon=True,
    fontsize=11
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()


# ============================================================
# 10. Save 300 DPI figure
# ============================================================

save_path = (
    "/kaggle/working/"
    "PCA_Real_DCGAN_cGAN_WGAN_Normalized_300dpi.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05,
    format="png"
)

plt.show()

print("Saved to:", save_path)

In [ ]:
WGAN_synthetic = WGAN_synthetic * 2 - 1

In [ ]:
print(f"Min: {WGAN_synthetic.min():.4f}")
print(f"Max: {WGAN_synthetic.max():.4f}")

if synthetic.min() >= -1 and synthetic.max() <= 1:
    print("✅ Images are normalized to [-1, 1].")
else:
    print("❌ Images are NOT in the range [-1, 1].")

In [ ]:
print(f"Min: {DCGAN_synthetic.min():.4f}")
print(f"Max: {DCGAN_synthetic.max():.4f}")

if synthetic.min() >= -1 and synthetic.max() <= 1:
    print("✅ Images are normalized to [-1, 1].")
else:
    print("❌ Images are NOT in the range [-1, 1].")

In [ ]:
DCGAN_synthetic = DCGAN_synthetic * 2 - 1

# DCGAN

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(DCGAN_synthetic), 20, replace=False)
samples = DCGAN_synthetic[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(8,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# WGAN

In [ ]:
# new
import matplotlib.pyplot as plt
import numpy as np

# Select 20 random fake images
idx = np.random.choice(len(WGAN_synthetic), 20, replace=False)
samples = WGAN_synthetic[idx]

# Optional: convert [-1,1] → [0,1] for better visualization
def denormalize(img):
    return (img + 1) / 2

# Plot (4 rows × 5 columns)
plt.figure(figsize=(8,6))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(denormalize(samples[i]).squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# cGAN pseudo mask

In [ ]:
pseudo_masks=model.predict(synthetic,batch_size=16)

print(pseudo_masks.shape)

# DCGAN pseudo mask

In [ ]:
import numpy as np

np.random.seed(42)  # For reproducibility

indices = np.random.choice(len(DCGAN_synthetic), 2000, replace=False)

synthetic2 = DCGAN_synthetic[indices]

print(synthetic2.shape)

In [ ]:
DCGAN_masks=model.predict(synthetic2,batch_size=16)

print(DCGAN_masks.shape)

In [ ]:
DCGAN_masks=(DCGAN_masks>0.5).astype(np.uint8)

In [ ]:
pseudo_masks=(pseudo_masks>0.5).astype(np.uint8)

In [ ]:
np.save("DCGAN_masks.npy",DCGAN_masks)

In [ ]:
np.save("DCGAN_synthetic_for_mask.npy",synthetic2)

# WGAN MASK

In [ ]:
WGAN_masks=model.predict(WGAN_synthetic,batch_size=16)

print(WGAN_masks.shape)

In [ ]:
WGAN_masks=(WGAN_masks>0.5).astype(np.uint8)

In [ ]:
np.save("WGAN_masks.npy",WGAN_masks)

# CGAN

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

indices = np.random.choice(len(synthetic), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = synthetic[idx].squeeze()
    mask = pseudo_masks[idx].squeeze()

    ax.imshow(img, cmap="gray")
    ax.imshow(mask, cmap="Reds", alpha=0.4)
    ax.set_title(f"Image {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

indices = np.random.choice(len(synthetic), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = synthetic[idx].squeeze()
    mask = pseudo_masks[idx].squeeze()

    ax.imshow(img, cmap="gray")
    ax.imshow(mask, cmap="Reds", alpha=0.4)
    ax.set_title(f"Image {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

indices = np.random.choice(len(synthetic), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = input_images[idx].squeeze()
    mask = input_masks[idx].squeeze().astype(np.uint8)

    ax.imshow(img, cmap='gray')

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Draw contours in red
    for cnt in contours:
        cnt = cnt.squeeze()
        if len(cnt.shape) == 2:
            ax.plot(cnt[:, 0], cnt[:, 1], color='red', linewidth=2)

    ax.set_title(f"Image {idx}", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Randomly choose 16 synthetic images
indices = np.random.choice(len(synthetic), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = synthetic[idx].squeeze()
    mask = pseudo_masks[idx].squeeze().astype(np.uint8)

    # Show synthetic MRI
    ax.imshow(img, cmap='gray')

    # Find tumor contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Draw contour
    for cnt in contours:
        cnt = cnt.squeeze()
        if len(cnt.shape) == 2:
            ax.plot(cnt[:, 0], cnt[:, 1], color='red', linewidth=2)

    tumor_pixels = int(mask.sum())

    ax.set_title(f"Img {idx}\nTumor: {tumor_pixels}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

# DCGAN

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

indices = np.random.choice(len(synthetic2), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = synthetic2[idx].squeeze()
    mask = DCGAN_masks[idx].squeeze()

    ax.imshow(img, cmap="gray")
    ax.imshow(mask, cmap="Reds", alpha=0.4)
    ax.set_title(f"Image {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

indices = np.random.choice(len(synthetic2), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = synthetic2[idx].squeeze()
    mask = DCGAN_masks[idx].squeeze()

    ax.imshow(img, cmap="gray")
    ax.imshow(mask, cmap="Reds", alpha=0.4)
    ax.set_title(f"Image {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Randomly choose 16 synthetic images
indices = np.random.choice(len(synthetic2), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = synthetic2[idx].squeeze()
    mask = DCGAN_masks[idx].squeeze().astype(np.uint8)

    # Show synthetic MRI
    ax.imshow(img, cmap='gray')

    # Find tumor contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Draw contour
    for cnt in contours:
        cnt = cnt.squeeze()
        if len(cnt.shape) == 2:
            ax.plot(cnt[:, 0], cnt[:, 1], color='red', linewidth=2)

    tumor_pixels = int(mask.sum())

    ax.set_title(f"Img {idx}\nTumor: {tumor_pixels}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Randomly choose 16 synthetic images
indices = np.random.choice(len(synthetic2), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = synthetic2[idx].squeeze()
    mask = DCGAN_masks[idx].squeeze().astype(np.uint8)

    # Show synthetic MRI
    ax.imshow(img, cmap='gray')

    # Find tumor contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Draw contour
    for cnt in contours:
        cnt = cnt.squeeze()
        if len(cnt.shape) == 2:
            ax.plot(cnt[:, 0], cnt[:, 1], color='red', linewidth=2)

    tumor_pixels = int(mask.sum())

    ax.set_title(f"Img {idx}\nTumor: {tumor_pixels}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

# WGAN

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

indices = np.random.choice(len(WGAN_synthetic), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = WGAN_synthetic[idx].squeeze()
    mask = WGAN_masks[idx].squeeze()

    ax.imshow(img, cmap="gray")
    ax.imshow(mask, cmap="Reds", alpha=0.4)
    ax.set_title(f"Image {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Randomly choose 16 synthetic images
indices = np.random.choice(len(WGAN_synthetic), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = WGAN_synthetic[idx].squeeze()
    mask = WGAN_masks[idx].squeeze().astype(np.uint8)

    # Show synthetic MRI
    ax.imshow(img, cmap='gray')

    # Find tumor contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Draw contour
    for cnt in contours:
        cnt = cnt.squeeze()
        if len(cnt.shape) == 2:
            ax.plot(cnt[:, 0], cnt[:, 1], color='red', linewidth=2)

    tumor_pixels = int(mask.sum())

    ax.set_title(f"Img {idx}\nTumor: {tumor_pixels}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Randomly choose 16 synthetic images
indices = np.random.choice(len(WGAN_synthetic), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = WGAN_synthetic[idx].squeeze()
    mask = WGAN_masks[idx].squeeze().astype(np.uint8)

    # Show synthetic MRI
    ax.imshow(img, cmap='gray')

    # Find tumor contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Draw contour
    for cnt in contours:
        cnt = cnt.squeeze()
        if len(cnt.shape) == 2:
            ax.plot(cnt[:, 0], cnt[:, 1], color='red', linewidth=2)

    tumor_pixels = int(mask.sum())

    ax.set_title(f"Img {idx}\nTumor: {tumor_pixels}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

#CGAN

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Randomly select 8 synthetic images
indices = np.random.choice(len(synthetic), 8, replace=False)

fig, axes = plt.subplots(8, 3, figsize=(10, 24))

for row, idx in enumerate(indices):

    # Image
    img = synthetic[idx].squeeze()

    # If image is normalized to [-1,1], convert for display
    img_display = (img + 1) / 2
    img_display = np.clip(img_display, 0, 1)

    # Mask
    mask = pseudo_masks[idx].squeeze().astype(np.uint8)

    # --------------------------
    # Column 1 : Synthetic MRI
    # --------------------------
    axes[row,0].imshow(img_display, cmap='gray')
    axes[row,0].set_title("Synthetic MRI")
    axes[row,0].axis('off')

    # --------------------------
    # Column 2 : Pseudo Mask
    # --------------------------
    axes[row,1].imshow(mask, cmap='gray')
    axes[row,1].set_title("Pseudo Mask")
    axes[row,1].axis('off')

    # --------------------------
    # Column 3 : Overlay
    # --------------------------
    axes[row,2].imshow(img_display, cmap='gray')

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        cnt = cnt.squeeze()
        if len(cnt.shape) == 2:
            axes[row,2].plot(cnt[:,0], cnt[:,1],
                             color='red',
                             linewidth=2)

    tumor_pixels = int(mask.sum())

    axes[row,2].set_title(f"Overlay ({tumor_pixels} px)")
    axes[row,2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Select 16 random synthetic images
indices = np.random.choice(len(synthetic), 16, replace=False)

fig, axes = plt.subplots(16, 3, figsize=(10, 48))

for row, idx in enumerate(indices):

    # Synthetic image
    img = synthetic[idx].squeeze()

    # If images are normalized to [-1,1], convert to [0,1] for display
    img_display = (img + 1) / 2
    img_display = np.clip(img_display, 0, 1)

    # Predicted mask
    mask = pseudo_masks[idx].squeeze().astype(np.uint8)

    # -----------------------
    # Column 1: Synthetic MRI
    # -----------------------
    axes[row, 0].imshow(img_display, cmap="gray")
    axes[row, 0].set_title(f"Image {idx}", fontsize=10)
    axes[row, 0].axis("off")

    # -----------------------
    # Column 2: Pseudo Mask
    # -----------------------
    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title("Pseudo Mask", fontsize=10)
    axes[row, 1].axis("off")

    # -----------------------
    # Column 3: Overlay
    # -----------------------
    axes[row, 2].imshow(img_display, cmap="gray")

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        cnt = cnt.squeeze()
        if len(cnt.shape) == 2:
            axes[row, 2].plot(
                cnt[:, 0],
                cnt[:, 1],
                color="red",
                linewidth=2
            )

    axes[row, 2].set_title(
        f"Tumor Pixels: {int(mask.sum())}",
        fontsize=10
    )
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

indices = np.random.choice(len(synthetic), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, idx in zip(axes.ravel(), indices):

    img = (synthetic[idx].squeeze() + 1) / 2
    img = np.clip(img, 0, 1)

    mask = pseudo_masks[idx].squeeze().astype(np.uint8)

    ax.imshow(img, cmap="gray")

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        cnt = cnt.squeeze()
        if len(cnt.shape) == 2:
            ax.plot(cnt[:, 0], cnt[:, 1], color="red", linewidth=2)

    ax.set_title(f"Sample {idx}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
raw_predictions = model.predict(synthetic, batch_size=16)

print("Mean confidence:", raw_predictions.mean())
print("Max confidence :", raw_predictions.max())
print("Min confidence :", raw_predictions.min())

In [ ]:
print("Average predicted tumor pixels:", tumor_area.mean())

# CGAN

In [ ]:
# Real tumor area distribution
real_tumor_area = input_masks.sum(axis=(1,2,3))

# Synthetic pseudo tumor area distribution
pseudo_tumor_area = pseudo_masks.sum(axis=(1,2,3))

print("===== Real BraTS Masks =====")
print(f"Mean tumor pixels : {real_tumor_area.mean():.2f}")
print(f"Median            : {np.median(real_tumor_area):.2f}")
print(f"Std               : {real_tumor_area.std():.2f}")
print(f"Min               : {real_tumor_area.min():.2f}")
print(f"Max               : {real_tumor_area.max():.2f}")


print("\n===== Synthetic Pseudo Masks =====")
print(f"Mean tumor pixels : {pseudo_tumor_area.mean():.2f}")
print(f"Median            : {np.median(pseudo_tumor_area):.2f}")
print(f"Std               : {pseudo_tumor_area.std():.2f}")
print(f"Min               : {pseudo_tumor_area.min():.2f}")
print(f"Max               : {pseudo_tumor_area.max():.2f}")

# DCGAN

In [ ]:
# Real tumor area distribution
real_tumor_area = input_masks.sum(axis=(1,2,3))

# Synthetic pseudo tumor area distribution
pseudo_tumor_area = DCGAN_masks.sum(axis=(1,2,3))

print("===== Real BraTS Masks =====")
print(f"Mean tumor pixels : {real_tumor_area.mean():.2f}")
print(f"Median            : {np.median(real_tumor_area):.2f}")
print(f"Std               : {real_tumor_area.std():.2f}")
print(f"Min               : {real_tumor_area.min():.2f}")
print(f"Max               : {real_tumor_area.max():.2f}")


print("\n===== DCGAN Synthetic Pseudo Masks =====")
print(f"Mean tumor pixels : {pseudo_tumor_area.mean():.2f}")
print(f"Median            : {np.median(pseudo_tumor_area):.2f}")
print(f"Std               : {pseudo_tumor_area.std():.2f}")
print(f"Min               : {pseudo_tumor_area.min():.2f}")
print(f"Max               : {pseudo_tumor_area.max():.2f}")

# WGAN

In [ ]:
# Real tumor area distribution
real_tumor_area = input_masks.sum(axis=(1,2,3))

# Synthetic pseudo tumor area distribution
pseudo_tumor_area = WGAN_masks.sum(axis=(1,2,3))

print("===== Real BraTS Masks =====")
print(f"Mean tumor pixels : {real_tumor_area.mean():.2f}")
print(f"Median            : {np.median(real_tumor_area):.2f}")
print(f"Std               : {real_tumor_area.std():.2f}")
print(f"Min               : {real_tumor_area.min():.2f}")
print(f"Max               : {real_tumor_area.max():.2f}")


print("\n===== DCGAN Synthetic Pseudo Masks =====")
print(f"Mean tumor pixels : {pseudo_tumor_area.mean():.2f}")
print(f"Median            : {np.median(pseudo_tumor_area):.2f}")
print(f"Std               : {pseudo_tumor_area.std():.2f}")
print(f"Min               : {pseudo_tumor_area.min():.2f}")
print(f"Max               : {pseudo_tumor_area.max():.2f}")

# CGAN

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(real_tumor_area, bins=50, alpha=0.5, label="Real Masks")
plt.hist(pseudo_tumor_area, bins=50, alpha=0.5, label="Synthetic Pseudo Masks")

plt.xlabel("Tumor pixels")
plt.ylabel("Number of images")
plt.title("cGAN Tumor Area Distribution Comparison")

plt.legend()
plt.show() 

# DCGAN

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(real_tumor_area, bins=50, alpha=0.5, label="Real Masks")
plt.hist(pseudo_tumor_area, bins=50, alpha=0.5, label="Synthetic Pseudo Masks")

plt.xlabel("Tumor pixels")
plt.ylabel("Number of images")
plt.title("DCGAN Tumor Area Distribution Comparison")

plt.legend()
plt.show()

# WGAN

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(real_tumor_area, bins=50, alpha=0.5, label="Real Masks")
plt.hist(pseudo_tumor_area, bins=50, alpha=0.5, label="Synthetic Pseudo Masks")

plt.xlabel("Tumor pixels")
plt.ylabel("Number of images")
plt.title("WGAN Tumor Area Distribution Comparison")

plt.legend()
plt.show()

# Step 5: Load Synthetic Images + Pseudo Masks

In [ ]:
synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/generated-cgan/generated_images_128 (8).npy"
)

pseudo_masks = np.load(
    "/kaggle/input/datasets/shantaakterid170/synthetic-mask/synthetic_masks.npy"
)

print(synthetic.shape)
print(pseudo_masks.shape)

# DCGAN

In [ ]:
synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/pseudo-mask/DCGAN_synthetic_for_mask.npy"
)

pseudo_masks = np.load(
    "/kaggle/input/datasets/shantaakterid170/pseudo-mask/DCGAN_masks.npy"
)

print(synthetic.shape)
print(pseudo_masks.shape)

# WGAN

In [ ]:
synthetic = np.load(
    "/kaggle/input/datasets/shantaakterid170/wgan-best/WGANgenerated_images7.npy"
)

pseudo_masks = np.load(
    "/kaggle/input/datasets/shantaakterid170/pseudo-mask/WGAN_masks.npy"
)

print(synthetic.shape)
print(pseudo_masks.shape)

In [ ]:
print(f"Min: {synthetic.min():.4f}")
print(f"Max: {synthetic.max():.4f}")

if synthetic.min() >= -1 and synthetic.max() <= 1:
    print("✅ Images are normalized to [-1, 1].")
else:
    print("❌ Images are NOT in the range [-1, 1].")

In [ ]:
synthetic = synthetic * 2 - 1

In [ ]:
from sklearn.utils import shuffle

X_aug = np.concatenate(
    [X_real10, synthetic],
    axis=0
)

Y_aug = np.concatenate(
    [Y_real10, pseudo_masks],
    axis=0
)

X_aug, Y_aug = shuffle(
    X_aug,
    Y_aug,
    random_state=42
)

print(X_aug.shape)

In [ ]:
aug_model = build_unet()

aug_model.compile(
    optimizer='adam',
    loss=bce_dice_loss,
    metrics=[
        dice_coef,
        tf.keras.metrics.BinaryIoU(
            target_class_ids=[1]
        )
    ]
)

history_aug = aug_model.fit(
    X_aug,
    Y_aug,

    validation_data=(
        X_val,
        Y_val
    ),

    epochs=50,
    batch_size=16,

    callbacks=callbacks
)

# Step 8: Evaluate Augmented Model

# cGAN

In [ ]:
aug_results = aug_model.evaluate(
    X_test,
    Y_test,
    verbose=0
)

print("===== Augmented U-Net =====")
print(f"Loss : {aug_results[0]:.4f}")
print(f"Dice : {aug_results[1]:.4f}")
print(f"IoU  : {aug_results[2]:.4f}")

# DCGAN

In [ ]:
aug_results = aug_model.evaluate(
    X_test,
    Y_test,
    verbose=0
)

print("===== Augmented U-Net =====")
print(f"Loss : {aug_results[0]:.4f}")
print(f"Dice : {aug_results[1]:.4f}")
print(f"IoU  : {aug_results[2]:.4f}")

# 2nd train

In [ ]:
aug_results = aug_model.evaluate(
    X_test,
    Y_test,
    verbose=0
)

print("===== Augmented U-Net =====")
print(f"Loss : {aug_results[0]:.4f}")
print(f"Dice : {aug_results[1]:.4f}")
print(f"IoU  : {aug_results[2]:.4f}")

# WGAN

In [ ]:
aug_results = aug_model.evaluate(
    X_test,
    Y_test,
    verbose=0
)

print("===== Augmented U-Net =====")
print(f"Loss : {aug_results[0]:.4f}")
print(f"Dice : {aug_results[1]:.4f}")
print(f"IoU  : {aug_results[2]:.4f}")

# Step 9: Final Comparison Table

In [ ]:
import pandas as pd

comparison = pd.DataFrame({

    "Training Data":[
        "320 Real",
        "320 Real + 2000 Synthetic"
    ],

    "Loss":[
        baseline_results[0],
        aug_results[0]
    ],

    "Dice":[
        baseline_results[1],
        aug_results[1]
    ],

    "IoU":[
        baseline_results[2],
        aug_results[2]
    ]
})

comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame({

    "Training Data":[
        "320 Real",
        "320 Real + DCGAN 2000 Synthetic"
    ],

    "Loss":[
        baseline_results[0],
        aug_results[0]
    ],

    "Dice":[
        baseline_results[1],
        aug_results[1]
    ],

    "IoU":[
        baseline_results[2],
        aug_results[2]
    ]
})

comparison

# 3200+2000 synthetic

In [ ]:
synthetic.shape
# (2000,128,128,1)

In [ ]:
pseudo_masks.shape
# (2000,128,128,1)

In [ ]:
X_augmented = np.concatenate(
    [X_train, synthetic[:1000]],
    axis=0
)

Y_augmented = np.concatenate(
    [Y_train, pseudo_masks[:1000]],
    axis=0
)


print("Augmented images:", X_augmented.shape)
print("Augmented masks :", Y_augmented.shape)

In [ ]:
from sklearn.utils import shuffle

X_augmented, Y_augmented = shuffle(
    X_augmented,
    Y_augmented,
    random_state=42
)

In [ ]:
aug_model = build_unet(
    input_shape=(128,128,1)
)

In [ ]:
aug_model.compile(
    optimizer=Adam(1e-4),
    loss=bce_dice_loss,
    metrics=[
        dice_coef,
        tf.keras.metrics.BinaryIoU(target_class_ids=[1])
    ]
)

In [ ]:
history_aug = aug_model.fit(

    X_augmented,
    Y_augmented,

    validation_data=(
        X_test,
        Y_test
    ),

    epochs=50,

    batch_size=16,

    callbacks=callbacks
)

# for 1000 synthetic

In [ ]:
history_aug = aug_model.fit(

    X_augmented,
    Y_augmented,

    validation_data=(
        X_test,
        Y_test
    ),

    epochs=50,

    batch_size=16,

    callbacks=callbacks
)

In [ ]:
results_aug = aug_model.evaluate(
    X_test,
    Y_test,
    verbose=0
)


for name,value in zip(
    aug_model.metrics_names,
    results_aug
):
    print(f"{name}: {value:.4f}")

In [ ]:
baseline_results = model.evaluate(
    X_test,
    Y_test,
    verbose=0
)

augmented_results = aug_model.evaluate(
    X_test,
    Y_test,
    verbose=0
)


import pandas as pd

comparison = pd.DataFrame({

    "Model":[
        "Baseline U-Net",
        "Synthetic Augmented U-Net"
    ],

    "Dice":[
        baseline_results[1],
        augmented_results[1]
    ],

    "IoU":[
        baseline_results[2],
        augmented_results[2]
    ]

})


comparison

In [ ]:
import pandas as pd

# Evaluate baseline U-Net
baseline_results = model.evaluate(
    X_test,
    Y_test,
    verbose=0
)

# Evaluate augmented U-Net
augmented_results = aug_model.evaluate(
    X_test,
    Y_test,
    verbose=0
)


# Create comparison table
comparison = pd.DataFrame({

    "Model": [
        "Baseline U-Net",
        "Synthetic Augmented U-Net"
    ],

    "Loss": [
        baseline_results[0],
        augmented_results[0]
    ],

    "Dice Coefficient": [
        baseline_results[1],
        augmented_results[1]
    ],

    "IoU": [
        baseline_results[2],
        augmented_results[2]
    ]

})


# Display with 4 decimal places
comparison.style.format({
    "Loss": "{:.4f}",
    "Dice Coefficient": "{:.4f}",
    "IoU": "{:.4f}"
})

# 1000 synthetic

In [ ]:
import pandas as pd

# Evaluate baseline U-Net
baseline_results = model.evaluate(
    X_test,
    Y_test,
    verbose=0
)

# Evaluate augmented U-Net
augmented_results = aug_model.evaluate(
    X_test,
    Y_test,
    verbose=0
)


# Create comparison table
comparison = pd.DataFrame({

    "Model": [
        "Baseline U-Net",
        "Synthetic Augmented U-Net"
    ],

    "Loss": [
        baseline_results[0],
        augmented_results[0]
    ],

    "Dice Coefficient": [
        baseline_results[1],
        augmented_results[1]
    ],

    "IoU": [
        baseline_results[2],
        augmented_results[2]
    ]

})


# Display with 4 decimal places
comparison.style.format({
    "Loss": "{:.4f}",
    "Dice Coefficient": "{:.4f}",
    "IoU": "{:.4f}"
})

# t-NSE


In [ ]:
# ==========================================================
# t-SNE Visualization (2000 Real vs 2000 Synthetic)
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ----------------------------------------------------------
# Load data
# ----------------------------------------------------------
real_images = input_images.squeeze()   # (4000,128,128)

fake_images = np.load(
    "/kaggle/input/datasets/shantaakterid170/dcgan-best-result/generated_images_DCGAN_BEST.npy"
).squeeze()

print("Original Real :", real_images.shape)
print("Original Fake :", fake_images.shape)

# ----------------------------------------------------------
# Use exactly 2000 samples from each
# ----------------------------------------------------------
N = 2000

# real_images = real_images[:N]
# fake_images = fake_images[:N]

np.random.seed(42)

real_idx = np.random.choice(len(real_images), N, replace=False)
fake_idx = np.random.choice(len(fake_images), N, replace=False)

real_images = real_images[real_idx]
fake_images = fake_images[fake_idx]

print("Real used :", real_images.shape)
print("Fake used :", fake_images.shape)

# ----------------------------------------------------------
# Flatten
# ----------------------------------------------------------
real_flat = real_images.reshape(N, -1)
fake_flat = fake_images.reshape(N, -1)

# ----------------------------------------------------------
# Combine
# ----------------------------------------------------------
X = np.vstack([real_flat, fake_flat])

labels = np.array([0]*N + [1]*N)

print("Combined:", X.shape)

# ----------------------------------------------------------
# Standardize
# ----------------------------------------------------------
scaler = StandardScaler()
X = scaler.fit_transform(X)

# ----------------------------------------------------------
# PCA (recommended before t-SNE)
# ----------------------------------------------------------
print("Running PCA...")

pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X)

print("PCA shape:", X_pca.shape)

# ----------------------------------------------------------
# t-SNE
# ----------------------------------------------------------
print("Running t-SNE...")

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42,
    max_iter=1000,
    verbose=1
)

X_tsne = tsne.fit_transform(X_pca)

print("t-SNE completed!")

# ----------------------------------------------------------
# Split
# ----------------------------------------------------------
real_tsne = X_tsne[:N]
fake_tsne = X_tsne[N:]

# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------
plt.figure(figsize=(10,8))

plt.scatter(
    real_tsne[:,0],
    real_tsne[:,1],
    c="royalblue",
    s=15,
    alpha=0.65,
    label="Real MRI"
)

plt.scatter(
    fake_tsne[:,0],
    fake_tsne[:,1],
    c="red",
    s=15,
    alpha=0.65,
    label="DCGAN"
)

plt.title("t-SNE: Real vs DCGAN Synthetic MRI", fontsize=16)
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Step 1: Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.applications.inception_v3 import (
    InceptionV3,
    preprocess_input
)

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

from tqdm import tqdm
import cv2

# Step 2: Load images

In [ ]:
real = input_images.squeeze()

dcgan = np.load("/kaggle/input/datasets/shantaakterid170/dcgan-best-result/generated_images_DCGAN_BEST.npy").squeeze()

cgan = np.load("/kaggle/input/datasets/shantaakterid170/generated-cgan/generated_images_128 (8).npy").squeeze()

wgan = np.load("/kaggle/input/datasets/shantaakterid170/wgan-best/WGANgenerated_images7.npy").squeeze()

# Step 3: Randomly sample 2000 images

In [ ]:
N = 2000

np.random.seed(42)

real = real[np.random.choice(len(real),N,replace=False)]

dcgan = dcgan[np.random.choice(len(dcgan),N,replace=False)]

cgan = cgan[np.random.choice(len(cgan),N,replace=False)]

wgan = wgan[np.random.choice(len(wgan),N,replace=False)]

In [ ]:
def to_01(img):
    # If image is in [-1,1]
    if img.min() < 0:
        img = (img + 1.0) / 2.0
    return np.clip(img, 0, 1)

In [ ]:
real = to_01(real)
dcgan = to_01(dcgan)
cgan = to_01(cgan)
wgan = to_01(wgan)

In [ ]:
import numpy as np

def check_intensity(name, images):
    print(f"\n{name}")
    print("-"*30)
    print("Shape :", images.shape)
    print("Min   :", np.min(images))
    print("Max   :", np.max(images))
    print("Mean  :", np.mean(images))
    print("Std   :", np.std(images))

# Check all datasets
check_intensity("Real", real)
check_intensity("DCGAN", dcgan)
check_intensity("CGAN", cgan)
check_intensity("WGAN-GP", wgan)

# Step 4: Convert MRI → Inception input

In [ ]:
def prepare_images(images):

    output = []

    for img in tqdm(images):

        img = img.squeeze()

        img = cv2.resize(img, (299, 299))

        # images are already in [0,1]
        img = img * 255.0

        img = np.stack([img, img, img], axis=-1)

        output.append(img)

    output = np.array(output, dtype=np.float32)

    return preprocess_input(output)

# Step 5: Prepare all images

In [ ]:
real299 = prepare_images(real)

dcgan299 = prepare_images(dcgan)

cgan299 = prepare_images(cgan)

wgan299 = prepare_images(wgan)

# Step 6: Load InceptionV3

In [ ]:
model = InceptionV3(
    include_top=False,
    pooling='avg',
    input_shape=(299,299,3)
)

# Step 7: Extract deep features

In [ ]:
print("Extracting Real Features...")
real_feat = model.predict(real299,batch_size=32,verbose=1)

print("Extracting DCGAN Features...")
dcgan_feat = model.predict(dcgan299,batch_size=32,verbose=1)

print("Extracting CGAN Features...")
cgan_feat = model.predict(cgan299,batch_size=32,verbose=1)

print("Extracting WGAN Features...")
wgan_feat = model.predict(wgan299,batch_size=32,verbose=1)

# Step 8: Combine

In [ ]:
X=np.vstack([
    real_feat,
    dcgan_feat,
    cgan_feat,
    wgan_feat
])

labels=np.array(
    ["Real"]*N+
    ["DCGAN"]*N+
    ["CGAN"]*N+
    ["WGAN-GP"]*N
)

print(X.shape)

# Step 9: Standardize

In [ ]:
scaler=StandardScaler()

X=scaler.fit_transform(X)

# Step 10: PCA

In [ ]:
print("Running PCA...")

pca=PCA(
    n_components=50,
    random_state=42
)

X_pca=pca.fit_transform(X)

print(X_pca.shape)

# Step 11: t-SNE

In [ ]:
print("Running t-SNE...")

tsne=TSNE(

    n_components=2,

    perplexity=30,

    learning_rate="auto",

    init="pca",

    random_state=42,

    max_iter=1000,

    verbose=1
)

X_tsne=tsne.fit_transform(X_pca)

# Step 12: Plot

In [ ]:
plt.figure(figsize=(11,9))

colors={

    "Real":"royalblue",

    "DCGAN":"red",

    "CGAN":"green",

    "WGAN-GP":"orange"

}

for label in np.unique(labels):

    idx=labels==label

    plt.scatter(

        X_tsne[idx,0],

        X_tsne[idx,1],

        s=12,

        alpha=0.55,

        c=colors[label],

        label=label
    )

plt.legend(fontsize=12)

plt.title("t-SNE of Real and Synthetic MRI Images",fontsize=16)

plt.xlabel("t-SNE Dimension 1")

plt.ylabel("t-SNE Dimension 2")

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

In [ ]:
# Variance explained by each principal component
explained_variance = pca.explained_variance_ratio_

print("First 10 principal components:")
for i, var in enumerate(explained_variance[:10], 1):
    print(f"PC{i}: {var*100:.2f}%")

# Cumulative variance
cumulative = np.cumsum(explained_variance)

print("\nCumulative explained variance:")
for i in [1, 2, 5, 10, 20, 30, 40, 50]:
    print(f"PC1-{i}: {cumulative[i-1]*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

cumulative = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8,5))
plt.plot(range(1, 51), cumulative * 100, marker='o')
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance (%)")
plt.title("PCA Explained Variance")
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# ===========================
# Split PCA embeddings
# ===========================
N = 2000

real_pca = X_pca[:N]
dcgan_pca = X_pca[N:2*N]
cgan_pca = X_pca[2*N:3*N]
wgan_pca = X_pca[3*N:4*N]

# ===========================
# Plot
# ===========================
plt.figure(figsize=(11,9))

plt.scatter(
    real_pca[:,0],
    real_pca[:,1],
    s=18,
    alpha=0.55,
    label="Real",
    color="royalblue"
)

plt.scatter(
    dcgan_pca[:,0],
    dcgan_pca[:,1],
    s=18,
    alpha=0.55,
    label="DCGAN",
    color="red"
)

plt.scatter(
    cgan_pca[:,0],
    cgan_pca[:,1],
    s=18,
    alpha=0.55,
    label="cGAN",
    color="green"
)

plt.scatter(
    wgan_pca[:,0],
    wgan_pca[:,1],
    s=18,
    alpha=0.55,
    label="WGAN-GP",
    color="orange"
)

plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0]*100:.2f}% variance)",
    fontsize=12
)

plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1]*100:.2f}% variance)",
    fontsize=12
)

plt.title("PCA Visualization of Real and Synthetic MRI Images", fontsize=15)

plt.legend(fontsize=11)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(12,10), dpi=300)

plt.scatter(real_pca[:,0], real_pca[:,1],
            c="#1f77b4", s=15, alpha=0.55, label="Real")

plt.scatter(dcgan_pca[:,0], dcgan_pca[:,1],
            c="#d62728", s=15, alpha=0.55, label="DCGAN")

plt.scatter(cgan_pca[:,0], cgan_pca[:,1],
            c="#2ca02c", s=15, alpha=0.55, label="cGAN")

plt.scatter(wgan_pca[:,0], wgan_pca[:,1],
            c="#ff7f0e", s=15, alpha=0.55, label="WGAN-GP")

plt.xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.2f}%)", fontsize=13)
plt.ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.2f}%)", fontsize=13)

plt.legend(frameon=True, fontsize=11)
plt.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12,10), dpi=300)

plt.scatter(real_pca[:,0], real_pca[:,1],
            c="#1f77b4", s=15, alpha=0.55, label="Real")

plt.scatter(dcgan_pca[:,0], dcgan_pca[:,1],
            c="#d62728", s=15, alpha=0.55, label="DCGAN")

plt.scatter(cgan_pca[:,0], cgan_pca[:,1],
            c="#2ca02c", s=15, alpha=0.55, label="cGAN")

plt.scatter(wgan_pca[:,0], wgan_pca[:,1],
            c="#ff7f0e", s=15, alpha=0.55, label="WGAN-GP")

plt.xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.2f}%)", fontsize=13)
plt.ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.2f}%)", fontsize=13)

plt.legend(frameon=True, fontsize=11)
plt.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()

# Save high-resolution PNG
plt.savefig(
    "/kaggle/working/PCA_Comparison_Real_DCGAN_cGAN_WGAN-GP.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ============================
# Function to extract image features
# ============================
def image_statistics(images):

    stats = {
        "mean": [],
        "std": [],
        "min": [],
        "max": [],
        "nonzero_ratio": []
    }

    for img in images:

        stats["mean"].append(np.mean(img))
        stats["std"].append(np.std(img))
        stats["min"].append(np.min(img))
        stats["max"].append(np.max(img))

        stats["nonzero_ratio"].append(
            np.sum(img != 0) / img.size
        )

    return stats


# ============================
# Calculate statistics
# ============================

dcgan_stats = image_statistics(dcgan)
cgan_stats  = image_statistics(cgan)
wgan_stats  = image_statistics(wgan)


# ============================
# Box Plot Function
# ============================

def plot_box(metric):

    plt.figure(figsize=(8,5))

    data = [
        dcgan_stats[metric],
        cgan_stats[metric],
        wgan_stats[metric]
    ]

    plt.boxplot(
        data,
        labels=[
            "DCGAN",
            "cGAN",
            "WGAN-GP"
        ]
    )

    plt.title(f"Distribution of {metric} for Generated MRI Images")
    plt.ylabel(metric)
    plt.grid(True)
    plt.show()


# Plot different artifact indicators

plot_box("mean")
plot_box("std")
plot_box("nonzero_ratio")

In [ ]:
# Example: Find images with highest intensity deviation

mean_values = np.array(cgan_stats["mean"])

outlier_index = np.argmax(
    np.abs(mean_values - np.mean(mean_values))
)

print("Most abnormal cGAN image index:", outlier_index)


plt.figure(figsize=(4,4))
plt.imshow(cgan[outlier_index], cmap="gray")
plt.title(f"cGAN Artifact Candidate #{outlier_index}")
plt.axis("off")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2


# ============================
# Select image index
# ============================
idx = 100   # change index


dc = dcgan[idx]
cg = cgan[idx]
wg = wgan[idx]


# ============================
# Artifact detection
# Compare DCGAN/WGAN with cGAN
# ============================

def get_difference(img1, img2):

    diff = np.abs(img1 - img2)

    # normalize
    diff = (diff - diff.min()) / (diff.max()-diff.min()+1e-8)

    # threshold high difference regions
    mask = (diff > 0.35).astype(np.uint8)

    # remove small noise
    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    return mask


dc_error = get_difference(dc, cg)
wg_error = get_difference(wg, cg)


# ============================
# Get bounding boxes
# ============================

def get_boxes(mask):

    contours,_ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    boxes=[]

    for c in contours:
        area=cv2.contourArea(c)

        if area > 20:
            x,y,w,h=cv2.boundingRect(c)
            boxes.append((x,y,w,h))

    return boxes



dc_boxes = get_boxes(dc_error)
wg_boxes = get_boxes(wg_error)



# ============================
# Draw boxes and arrows
# ============================

def show_result(img, boxes, title):

    plt.figure(figsize=(5,5))

    plt.imshow(img,cmap="gray")


    for (x,y,w,h) in boxes:

        # artifact box
        rect = plt.Rectangle(
            (x,y),
            w,
            h,
            fill=False,
            linewidth=2,
            edgecolor="red"
        )

        plt.gca().add_patch(rect)


        # arrow pointing to artifact
        plt.arrow(
            x+w/2,
            y-15,
            0,
            10,
            width=0.8,
            head_width=5,
            head_length=5
        )


    plt.title(title)
    plt.axis("off")
    plt.show()



# ============================
# Display
# ============================

plt.figure(figsize=(16,4))

images=[
    dc,
    cg,
    wg
]

titles=[
    "DCGAN\nArtifact Regions",
    "cGAN\nCleaner Tumor Structure",
    "WGAN-GP\nArtifact Regions"
]


for i,(img,title) in enumerate(zip(images,titles)):

    plt.subplot(1,3,i+1)

    plt.imshow(img,cmap="gray")


    if i==0:
        for x,y,w,h in dc_boxes:
            plt.gca().add_patch(
                plt.Rectangle(
                    (x,y),w,h,
                    fill=False,
                    linewidth=2,
                    edgecolor="red"
                )
            )

    if i==2:
        for x,y,w,h in wg_boxes:
            plt.gca().add_patch(
                plt.Rectangle(
                    (x,y),w,h,
                    fill=False,
                    linewidth=2,
                    edgecolor="red"
                )
            )


    plt.title(title)
    plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt


def edge_image(img):

    img = ((img+1)*127.5).astype(np.uint8)

    edges = cv2.Canny(
        img,
        50,
        150
    )

    return edges



idx=50

plt.figure(figsize=(12,3))

for i,img,title in zip(
    range(4),
    [real[idx], dcgan[idx], cgan[idx], wgan[idx]],
    ["Real","DCGAN","cGAN","WGAN-GP"]
):

    plt.subplot(1,4,i+1)

    plt.imshow(
        edge_image(img),
        cmap="gray"
    )

    plt.title(title)
    plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import f_oneway

dcgan_fid = [89.59,84.71,60.49]
cgan_fid  = [34.06,34.06,52.54]
wgan_fid  = [90.35,97.30,76.26]

F,p = f_oneway(dcgan_fid,cgan_fid,wgan_fid)

print("FID ANOVA")
print("F =",F)
print("p =",p)

In [ ]:
dcgan_is = [2.3527,2.2643,2.1287]
cgan_is  = [2.2573,2.2573,2.1629]
wgan_is  = [2.0636,2.1064,2.1967]

F,p = f_oneway(dcgan_is,cgan_is,wgan_is)

print("IS ANOVA")
print("F =",F)
print("p =",p)

In [ ]:
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd

scores = dcgan_fid + cgan_fid + wgan_fid

models = (["DCGAN"]*3 +
          ["cGAN"]*3 +
          ["WGAN-GP"]*3)

df = pd.DataFrame({
    "FID":scores,
    "Model":models
})

tukey = pairwise_tukeyhsd(
    endog=df["FID"],
    groups=df["Model"],
    alpha=0.05
)

print(tukey)

In [ ]:
scores = dcgan_is + cgan_is + wgan_is

df = pd.DataFrame({
    "IS":scores,
    "Model":models
})

tukey = pairwise_tukeyhsd(
    endog=df["IS"],
    groups=df["Model"],
    alpha=0.05
)

print(tukey)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Data
data = pd.DataFrame({
    "Model": (
        ["DCGAN"]*3 +
        ["cGAN"]*3 +
        ["WGAN-GP"]*3
    ),
    "FID": [
        89.59,84.71,60.49,
        34.06,34.06,52.54,
        90.35,97.30,76.26
    ]
})


plt.figure(figsize=(8,6))

palette = {
    "DCGAN":"#66b3ff",
    "cGAN":"#66cc66",
    "WGAN-GP":"#ff9999"
}


ax = sns.boxplot(
    data=data,
    x="Model",
    y="FID",
    palette=palette,
    width=0.5
)


sns.stripplot(
    data=data,
    x="Model",
    y="FID",
    color="black",
    size=8,
    jitter=True
)


# ==============================
# Significance markers
# ==============================

def add_significance(x1, x2, y, text):

    ax.plot(
        [x1, x2],
        [y, y],
        linewidth=1.5,
        color="black"
    )

    ax.text(
        (x1+x2)/2,
        y+2,
        text,
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold"
    )


# DCGAN vs cGAN
add_significance(
    0,
    1,
    108,
    "*"
)


# cGAN vs WGAN-GP
add_significance(
    1,
    2,
    120,
    "**"
)


# Formatting
plt.ylabel(
    "FID Score (Lower is Better)",
    fontsize=12
)

plt.xlabel(
    "GAN Model",
    fontsize=12
)

# plt.title(
#     "Distribution of FID Scores Across Independent Runs",
#     fontsize=14,
#     fontweight="bold"
# )


plt.ylim(0,140)


# cleaner look
sns.despine()

plt.grid(
    axis="y",
    linestyle="",
    alpha=0.3
)



plt.tight_layout()

save_path = "/kaggle/working/FID_Distribution_Across_GAN_Models_300dpi.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05,
    format="png"
)

plt.show()

print("Saved to:", save_path)

In [ ]:
from scipy.stats import f_oneway

dcgan_is = [2.3527, 2.2643, 2.1287]
cgan_is  = [2.2573, 2.2573, 2.1629]
wgan_is  = [2.0636, 2.1064, 2.1967]

F, p = f_oneway(
    dcgan_is,
    cgan_is,
    wgan_is
)

print("IS ANOVA")
print("F =", F)
print("p =", p)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Data
data = pd.DataFrame({
    "Model": (
        ["DCGAN"]*3 +
        ["cGAN"]*3 +
        ["WGAN-GP"]*3
    ),
    "IS": [
        2.3527, 2.2643, 2.1287,
        2.2573, 2.2573, 2.1629,
        2.0636, 2.1064, 2.1967
    ]
})


plt.figure(figsize=(8,6))

palette = {
    "DCGAN":"#66b3ff",
    "cGAN":"#66cc66",
    "WGAN-GP":"#ff9999"
}


ax = sns.boxplot(
    data=data,
    x="Model",
    y="IS",
    palette=palette,
    width=0.5
)


sns.stripplot(
    data=data,
    x="Model",
    y="IS",
    color="black",
    size=8,
    jitter=True
)


# ==============================
# Significance markers
# ==============================

def add_significance(x1, x2, y, text):

    ax.plot(
        [x1, x2],
        [y, y],
        linewidth=1.5,
        color="black"
    )

    ax.text(
        (x1+x2)/2,
        y+0.02,
        text,
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold"
    )


# Add these only if supported by your
# post-hoc statistical test

# Example:
# add_significance(0, 1, 2.40, "*")
# add_significance(1, 2, 2.45, "**")


# Formatting
plt.ylabel(
    "Inception Score (Higher is Better)",
    fontsize=12
)

plt.xlabel(
    "GAN Model",
    fontsize=12
)

plt.title(
    "Distribution of Inception Scores Across Independent Runs",
    fontsize=14,
    fontweight="bold"
)

plt.ylim(1.9, 2.6)

sns.despine()

plt.grid(
    axis="y",
    linestyle="",
    alpha=0.3
)

plt.tight_layout()


# Save 300 DPI
save_path = "/kaggle/working/IS_Distribution_Across_GAN_Models_300dpi.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05,
    format="png"
)

plt.show()

print("Saved to:", save_path)

In [ ]:
import matplotlib.pyplot as plt

# Data from the figure
years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
publications = [4, 8, 12, 20, 35, 60, 85, 120, 170, 156, 110]

# Create figure
plt.figure(figsize=(10, 6), dpi=300)

# Bar chart
plt.bar(
    years,
    publications,
    width=0.6,
    color='#1976D2',
    edgecolor='none'
)

# Axis labels
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Publications', fontsize=12)

# Axis limits and ticks
plt.xticks(years, fontsize=10)
plt.yticks(range(0, 181, 20), fontsize=10)
plt.ylim(0, 180)

# Grid
plt.grid(
    axis='y',
    linestyle='--',
    linewidth=0.6,
    alpha=0.5
)

# Keep grid behind bars
plt.gca().set_axisbelow(True)

# Adjust layout
plt.tight_layout()

# Save as 300 DPI PNG
output_path = '/kaggle/working/publications_by_year_300dpi.png'

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)

plt.show()

print("Saved:", output_path)

In [ ]:
import matplotlib.pyplot as plt

# Data
years = list(range(2015, 2026))

image_generation = [5, 8, 11, 18, 30, 50, 70, 95, 120, 110, 90]
image_translation = [3, 5, 8, 15, 25, 40, 60, 80, 100, 95, 75]
data_augmentation = [2, 3, 6, 10, 18, 30, 45, 60, 75, 70, 55]

# Create figure
plt.figure(figsize=(10, 6), dpi=300)

# Plot lines
plt.plot(
    years, image_generation,
    marker='o', linewidth=1.8,
    label='Image Generation'
)

plt.plot(
    years, image_translation,
    marker='o', linewidth=1.8,
    label='Image Translation'
)

plt.plot(
    years, data_augmentation,
    marker='o', linewidth=1.8,
    label='Data Augmentation'
)

# Labels
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Papers', fontsize=12)

# X-axis
plt.xticks(years[1::2], fontsize=10)

# Y-axis
plt.yticks(range(0, 121, 20), fontsize=10)
plt.ylim(0, 125)

# Grid
plt.grid(
    True,
    linestyle='-',
    linewidth=0.6,
    alpha=0.5
)

# Legend
plt.legend(
    loc='upper left',
    fontsize=10,
    frameon=True
)

# Layout
plt.tight_layout()

# Save 300 DPI
output_path = '/kaggle/working/research_trend_300dpi.png'

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)

plt.show()

print("Saved:", output_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Data
data = pd.DataFrame({
    "Model": (
        ["DCGAN"]*3 +
        ["cGAN"]*3 +
        ["WGAN-GP"]*3
    ),
    "FID": [
        89.59,84.71,60.49,
        34.06,34.06,52.54,
        90.35,97.30,76.26
    ]
})

plt.figure(figsize=(8,6))

palette = {
    "DCGAN":"#66b3ff",
    "cGAN":"#66cc66",
    "WGAN-GP":"#ff9999"
}

sns.boxplot(
    data=data,
    x="Model",
    y="FID",
    palette=palette,
    width=0.5
)

sns.stripplot(
    data=data,
    x="Model",
    y="FID",
    color="black",
    size=8,
    jitter=True
)

plt.ylabel("FID Score")
plt.title("Distribution of FID Scores Across Independent Runs")

plt.show()

In [ ]:
import numpy as np

means = [
    np.mean([89.59,84.71,60.49]),
    np.mean([34.06,34.06,52.54]),
    np.mean([90.35,97.30,76.26])
]

stds = [
    np.std([89.59,84.71,60.49],ddof=1),
    np.std([34.06,34.06,52.54],ddof=1),
    np.std([90.35,97.30,76.26],ddof=1)
]

models=["DCGAN","cGAN","WGAN-GP"]

plt.figure(figsize=(8,6))

colors=["#66b3ff","#66cc66","#ff9999"]

plt.bar(
    models,
    means,
    yerr=stds,
    capsize=8,
    color=colors
)

plt.ylabel("Mean FID")

plt.title("FID Score (Mean ± SD)")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

data = pd.DataFrame({
    "FID": dcgan_fid + cgan_fid + wgan_fid,
    "Model": (
        ["DCGAN"]*3 +
        ["cGAN"]*3 +
        ["WGAN-GP"]*3
    )
})


plt.figure(figsize=(7,5))

sns.boxplot(
    x="Model",
    y="FID",
    data=data
)

sns.stripplot(
    x="Model",
    y="FID",
    data=data,
    size=8
)

plt.ylabel("FID Score (Lower is Better)")
plt.xlabel("GAN Model")

plt.title(
    "Comparison of FID Scores Among GAN Models"
)

plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


# ==========================
# Prepare Data
# ==========================

df = pd.DataFrame({
    "FID": dcgan_fid + cgan_fid + wgan_fid,
    "Model": (
        ["DCGAN"]*3 +
        ["cGAN"]*3 +
        ["WGAN-GP"]*3
    )
})


# ==========================
# Plot
# ==========================

plt.figure(figsize=(8,6))

palette = {
    "DCGAN": "#4C72B0",
    "cGAN": "#55A868",
    "WGAN-GP": "#C44E52"
}


ax = sns.boxplot(
    data=df,
    x="Model",
    y="FID",
    palette=palette,
    width=0.5
)


# show individual runs
sns.stripplot(
    data=df,
    x="Model",
    y="FID",
    color="black",
    size=8,
    jitter=True
)


plt.ylabel("FID Score (Lower is Better)", fontsize=12)
plt.xlabel("GAN Model", fontsize=12)

plt.title(
    "Statistical Comparison of FID Scores Among GAN Models",
    fontsize=14,
    fontweight="bold"
)


# ==========================
# Tukey Significance Lines
# ==========================

def add_significance(ax, x1, x2, y, h, text):

    ax.plot(
        [x1, x1, x2, x2],
        [y, y+h, y+h, y],
        lw=1.5,
        c="black"
    )

    ax.text(
        (x1+x2)/2,
        y+h,
        text,
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold"
    )


# Highest y position
y_max = df["FID"].max()

# DCGAN vs cGAN
add_significance(
    ax,
    0,
    1,
    105,
    3,
    "*"
)

# cGAN vs WGAN-GP
add_significance(
    ax,
    1,
    2,
    115,
    3,
    "**"
)

# DCGAN vs WGAN-GP
add_significance(
    ax,
    0,
    2,
    125,
    3,
    "ns"
)


plt.ylim(0,140)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ===============================
# Data
# ===============================

models = ["DCGAN", "cGAN", "WGAN-GP"]

fid_scores = [
    dcgan_fid,
    cgan_fid,
    wgan_fid
]

means = [np.mean(x) for x in fid_scores]
stds  = [np.std(x) for x in fid_scores]


# ===============================
# Plot
# ===============================

fig, ax = plt.subplots(figsize=(7,5))


bars = ax.bar(
    models,
    means,
    yerr=stds,
    capsize=6,
    width=0.55
)


# Add values on top of bars

for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x()+bar.get_width()/2,
        mean+stds[bars.index(bar)]+3,
        f"{mean:.2f}",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )


# ===============================
# Tukey Significance Markers
# ===============================

def significance_line(x1,x2,y,text):

    ax.plot(
        [x1,x1,x2,x2],
        [y,y+2,y+2,y],
        linewidth=1.5,
        color="black"
    )

    ax.text(
        (x1+x2)/2,
        y+3,
        text,
        ha="center",
        fontsize=14,
        fontweight="bold"
    )


# significant only

# DCGAN vs cGAN
significance_line(
    0,
    1,
    115,
    "*"
)

# cGAN vs WGAN-GP
significance_line(
    1,
    2,
    130,
    "**"
)


# ===============================
# Formatting
# ===============================

ax.set_ylabel(
    "FID Score (Lower is Better)",
    fontsize=12
)

ax.set_xlabel(
    "GAN Architecture",
    fontsize=12
)

ax.set_title(
    "Comparison of FID Scores Between GAN Models",
    fontsize=14,
    fontweight="bold"
)


ax.set_ylim(0,150)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)


plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ===============================
# Data
# ===============================

models = ["DCGAN", "cGAN", "WGAN-GP"]

fid_scores = [
    dcgan_fid,
    cgan_fid,
    wgan_fid
]

means = [np.mean(x) for x in fid_scores]
stds  = [np.std(x) for x in fid_scores]


# ===============================
# Plot
# ===============================

fig, ax = plt.subplots(figsize=(7,5))


# Different colors
colors = [
    "#4C78A8",   # DCGAN blue
    "#59A14F",   # cGAN green
    "#E15759"    # WGAN red
]


bars = ax.bar(
    models,
    means,
    yerr=stds,
    capsize=5,
    width=0.55,
    color=colors,
    edgecolor="black",
    linewidth=1
)


# Add mean values

for i, bar in enumerate(bars):

    ax.text(
        bar.get_x()+bar.get_width()/2,
        means[i]+stds[i]+3,
        f"{means[i]:.2f}",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )


# ===============================
# Tukey Significance Brackets
# (No vertical lines)
# ===============================

def add_sig(ax, x1, x2, y, text):

    ax.plot(
        [x1, x2],
        [y, y],
        linewidth=1.5,
        color="black"
    )

    ax.text(
        (x1+x2)/2,
        y+3,
        text,
        ha="center",
        fontsize=14,
        fontweight="bold"
    )


# DCGAN vs cGAN
add_sig(
    ax,
    0,
    1,
    112,
    "*"
)


# cGAN vs WGAN-GP
add_sig(
    ax,
    1,
    2,
    130,
    "**"
)


# ===============================
# Formatting
# ===============================

ax.set_ylabel(
    "FID Score (Lower is Better)",
    fontsize=12
)

ax.set_xlabel(
    "GAN Model",
    fontsize=12
)


ax.set_title(
    "FID Score Comparison of GAN Architectures",
    fontsize=14,
    fontweight="bold"
)


ax.set_ylim(0,150)


# cleaner background
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


ax.grid(
    axis="y",
    linestyle="",
    alpha=0.3
)


plt.tight_layout()

plt.show()